# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
default_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = f'ids_{ID_FROM}_{ID_TO}_synced'
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO6TW{RP6@J&Rn8+<irn}T7j?qe58IJ8BM&iV9oS=ZRU`FKZ?nJzKNJ`pB0$SvuPX!vJKoJxH;siyI21Qe}Pu4>p6y$%Xzog%p8FINd$th~U(k^*!bMEJiR##Ww84O~X3MSK143#X$Yhi|uX~`nVhiO)nGAtNRBm8Go8uNV-$0A}%97=J-ijfc{8)S-c7Ak%eGgYPj>gwu>O!7>XjH_YJRUuXeDoa?H0XZyXmKMwc+|4TRQ#JhXW0}{VN4yxta=-anCNblM{vC)S<hiI{eN<$r;gIu^aBAw0@U;d~I7OB;-x}m%T+|<{REAk3B3|+pLoYrC)0O}=WIpCfWk)^^_q2ivAXol$Wo0E21CZvCOM-Y?D-8dXC@Ym}A^kYxQK4V@1QwLyxO7Ddt}-3=oT?mjf9ZIhr4(jK&co6rsA*yjbNr7onYt~J{WMbvkLAb0b>4pcuG3)}&oA;=mTp6|7;%@trwLkR`*^$g>MaA{kHm2#hoS&>eKvj+Xakf=2zNh=#vN9RZwW?13&eViUi;>2bH)yHl*nTq3dhsna=_B8tg-x}R5Euhsg`_iXQ=?LNb+(VkQ7lCRtX%{(U7GY&!Lc<K3S1yrbE}V@H(29h8D-6$V=vGP(f$%ACbGN`>?L+u=jaf3H^DgL{+LCobShCMlPeVgu_Y}TSF>nrVb1Ljv8uc!xbVHA<j<`(hG=G5O^(2pGQ%^EsE=OyIEDj%t}11q!Ll@iK@hv@Fr}#1$>0uW1Nc~BEJLe4|o-qy^ZztEAfl0QX!#^M4b1W_l1%J30s@z2saZAO2k>nOT5rQgw&D65e{C6qQKFt<w3ZBE-Wkki<lLzd5yE-Ek&?PPknqhFA|>00h|(Vb*K^G8guH$J_XvuxyapXkb)<9EHpsucqRh($E5&EkG<4uDDPRwz(j-7ZyBG<LbUwCb>7vWI)GRri-ecq$g)3@<%lg|p5-E)k9<lba96>T2}^^B)9$rW2xV<gXI`7XEgBt}p45TVuk{dv-HF<BWaRWUp3+E~_L)9CL74B}Rp-4|A)k<SYr!7eE9`m2agOw^W4~s##qgn0@dBnA#duCY8F9+m)}w&ZGK%rOeq-Cl<U;}Xw4|)fT4yjh4CH5$oYz2TL$yRP)^XJSVvEo}I?X=Z{D4j*QNrAqC;JiKve4SQ+>Q0ko9tyqzmRwKowiuh;`vpMq>G$NG$)ebl=giq8PE9Y)Vcf93>Nt7bgVA&TdRD;=2KgyD80EY1$Ed!#yE7*+jGgzNQ5%Q&sajf+6_Ff@A+jG<YWEVYrCSM)%ab5LKeT$cQoD^;ng@V5S&pa3SDOODQD7^J+wl3P!VOMJyXF=c;$wx0;496SWraqFwXW}=VhC7Y_y?dS3JHZ2rP}WS}7K(buLl`>Bd6N>*}h5Kd#`FDK!wWD0LiBV~1K)pv!4EAau9c5tp`yi(?j!M0k*+vncDJ4mr{`X{Fx|SgIrR^u!!s_E;0q2%X?a5n2zV5UQI{$3!o9@D#^K0~!=1FROyynXsODjxcftF&`4dUjM8eC=T^Ij+R2Vtd#n}KtJvKI@b!D>Fwhe4VuB~6F^VL%2s3Ev(>41wQ|?;vR)5;%W<hVsoc?$kt<p->KQ1aW;4SA86HZM&vZyl7(*eCa3XP?6wAm{Zw9kazpY!eBv&ZTwTFXbgV>)blv8AEfhB6SUI}rrT9+K_F^=;Bnr$_Uy0L_QRasExcK4m1Kf3pNcTsYMj;K#C%D_Qmjz%e{>#n2ojV-=&xYQ4JOn(xP%Mqc){jWEN3rQ!4^J3MJv1fg`@ADibhvtSF$y<oj4!=FCr9K36TvDUDJpWTVTss(D{0n}fX|W$Z{qg(D^RGj8@12L+>{ukN7-xqUUut(6UY>s%`rf<_L|ll);^XkrBnF|wUpFDTh1S*3iR2;FOJQk8%C3lHnW=7$sZvL&D~wrSl%lYvaGH6gzmO}0R@qPEiXLC(|2xMp7HD95%`j$tNoSP~p?k{(ZJ{+2oA7n!vEsSj<QV2`OXwMIvF6v1K{$ee9kM=^1^w(QT!Tb$F4UojG@_9byv-@0H}izdJ{gl1IJW_#TxLE|Sk`0~`^!?7w}OE}FsR~q+!E1S#@;yJaz;oEGSp48f!tf~Un7*&tnPGR9Bd^T&01pqG37(o@z>Tg)H2J@TPxwi>uJhlcoju7^#|JTx$(b|5(CGxyBBD_CTPNdn;@8iy-owXEO|JhQ7=GEHjcGdX$}=}FcMH{!E_<G3P-MTKJeW<8k}NOJ-&T<92dv8J@@w3&rbJl-{^aYkBY`q|G^c^Z9}L%$74He<JEQTSIbqn-0WBQm?oej&yLGQkcBER6S1>g?>E?kG!jIFwqmB^hdPC#>TBj(N}x!t9hQniOu0jm3N@L2%aO3rMcn<RjC2}^+oqiOhS>_7(x-A93;G}jiYNLLwsiXSqVhML%Kb=XxhY-C54^O(e304<R@TF%mbskuKE2b@?$-b==C-`HTwUqjH03S4_$M|Bv`~Z;uU$TWhR*&Wm)q<>UY>sf#VbCkZDfXw)lE&i(vi567GaXsr<$pj$3K)IvUD~mAJF1573KZoJkFF*r%91@jeY1-X+G|}^uZpSM-EPR^n=~g9qbf$?RL&a$Fvp}dvG=wNa!O|LBiERl?U*t17}gsH`_5&W9IOXR$a)cSMh>G#A}9*dD}s|-7@49BJ^iRqymUjuVzq+`aJZ-4j2^Gl-ler*d1uU3Uxiv!v}pG=&uv9W*DgwC}XVW3*!oO!1gAiLydqLg6cQ*JcU=*#}dedZZJ5s`z!#`Dy=i?o4U6}ZZdg<dRAnSStRC6R_8%IsYDVjfH|l)Pqy9}76G-hrn&<Or*lJNirLzY@G=7})#w>@y&!`Ywd+~Y$$m1Mc1bO!nJlQb9k%YVUABHbn<B(|2K1aeRhcC^)50)oNWtw}i*2nR8g=Pj0SicGhcIj)nRRX7nWsLguv)QoA!z%~x*0a>B>=C`&IZ;4?E=6*!X1h!Gp=6NMOBg)t_7k|p{NwLoJA;Q?{$v+*TF%lvh+5+$tJy72QFY}f~ejiw)i<VJGc0%MI1f$3!Td+GrQIE683fCn8diN=H~sYvQMv_eZb~GjC*ER##FNrsCO`0>$q7d^sq%YHvVJ%%e}dYMqLNBTT|;`WK7Mz&|=bNyJH$G3**@YWUi`utXbh`cr3iFF58z88WzbATdSgd7Xe)-x(B;{n^DR#7Lz50`AR9LnX9+vNA%8uVu<vmpyU!DLn3Ie9csr+F|`Nl0k&+Lt(Vkp&=z>^-694F=CY8-L0TpI*yv5=1xwIqOQ^{cs7c4P<D9jfp)ud2-!sbmzc{Ma|58UK1(9)_otKb^=yF3}0qySSo5Pp`e)UkU?(QU_#C0VNPonItY7%&=O|*N5f(#`|<yEn>W}eL?3C9+4n*L7)7~{4!Zso^xj|lO)mE1<G4`sTwc{4xWCM$GDLJmg&^KYF+;4S0gWFJu@)Yb+eX0eRejW7(ib6mLKTw5EPK(%D(Gjb0uNTfQ6BxY4U-ojmNv`s&{;E5-t=;Cs<O44F$W1txRZx4B{ML2^!PmV^=UAMp_Dz@;wd!%^2ouPCM;_OI&+v4f?XoR+Q+Z^d;`$%U#6|xhs$kf)2U)_4;Y{zt>^ST*r>^>UF%jf^D+I<+}jFlH(N(>Q~=fA6(q_VRH_?>+B;pOx1DwYn-uaUg?E0(STN$2LK&+cBHe_n#j#WUbzVWB&=^78yU>blX`U3`_YeIkxX>ab8l=cCK#UmI{^OrQPmhl{Vqu=|hS^ZNAx<SbDP@C7d}pZ_Hx^lvyMp{2<e65ErDXX%Iy<;639XD#1#`m_7a#^on@0=IZ{+c3GP_U(Oj4M471H$sEe9qSakJGM~lHv2F3pWpnEodDaNN+f3QKt`|#P@#dc8HnEHyF1v1Fum*OIpjVLIGFo)1jQ@6q`h=lU7ml2&B#QJ9jhbs&7URAu1`Ez>J_k2!Z2!g=UTE!F8sAfCVjUNlh7b+&ATncU2Pp~W(*(1{6vyjrt_<pdD?3@GySwGU7h5HrjrI$$_q2)wN(-5^-OVXh99D|GzBG*PCz%<n-ka1?4MW|H`H?D_GGn-<@?%ukKbYkqs#L@=d~fQ`)v29XKN>|X7<cqZHJrFWwi7=tV5#N^tA%--8b%Zv5K8;bcp+XtM4IT{|;2RJuc=dvfAf4)A_=r3wC0iZ11`@vJYvmge8a8fl8yBHTd+ANV3;zh<$`5<~$(c=NnHRzSUCFGCtEiXa8)Te4U+iJJqPWiDK_;VWQf$K|yjg$-$%yuppbZ4RiP6S%@5S@onC=MSAh&m>tur6o`LIS+0<RQ%;KnG9nP3;gW$Er!3KFtUwI07+YN$Ywg$zfB6=_(fW+YZqL6NTl1wrku0k45~}+Le(AP1NBqNS3*VWO^OSPy7wnY1OMKerJ|*B&*6ntg{<(!Gee2`(m^c8y3A(v6fSl+i9-mCP#?y`&>{{1nX8!SY6?f~tY(j320z+qy59eCl5A_j0(Q15fZ<~EM6Rq`o`uY2?$!dp9`bS=WPKi7n@oF1jIOT8Cc6KM&Y>KX7Y_!&&&J}mwyZ7Kp@c90tcOF0SlW1A4NTTJ<3jQSJ1!e^O7ajD+H|R`*AjQTf2y{ao1oVd`L14#uRBG&6B8cZx>PQE22qNy1ET#V>W#zxxsz);',
 'corpus_runner_snapshot.py': 'c-o~K+iu*(^?ko$syvYEtSC9M-Bb=A6kAD*IF>8R35xB4p~%_YiMS-gAtkG_{-J#+`r05U0<;KP^wECkgM<8s{-o#3g`DAKB_#^r-JQ#sGv|JpY;0^CRw*f$l<;L((3r&8Dr70ii|m*Znl4yMISJF4<kfM)_>w$%yqOo2)8dq+3le2TUU3pFX>^ijEG@aWv9U2?t2`@85*CX*EI1v_i);mJlY~Yk%Ti9%f=5}Emb93Vn9jp0DPtCudUM22^;h|kuw$<(SyGQ?`r9`=OZE3vST5yxIeddX`mE<Tp}zBTJ{o<wfB53b!876$M_yyIsM3@co2NU@=<)t1doQ0Jy@1(^5rIFaf;$J{f~KXjOPnvVKd~eUx9)q}#Qln;adyVZvm<hM+uMFfVBo>Mhve0Rdt<Vf=L!9a9)Hftt@}H_^>!W*_w&z=zIZwz2|J<WQy^V7CXbeYGu^to0|4+p@*<pv1yfs`i9{l%Nq7$D-}NTM3CohM@{+^v9r#(Y70oKdZu-GC%-_53&0zSriWd||?|ulQ5)3df2UY~Y<xK4VBuqHflVUK)a=|K?ywF3f1J!7h0o4FlbN=1?>Y$!Ky{n*MY;4&<jGMW#I1g&Xe`n6kLzYwp4azc++b$K_#|5weShUJwiX6g)G!C*fId>+_w8h3}SxR9p60{GFSsE7Sdf0}VWEQ$WTLb1<$jX3IAVkd7^_xk9Xdg^Qm!r`Lgq?`%^Kb>S!QH3`&k|Gv06yLo2d~1E%_%QI{?x1|P(V?4VHw#Xj$Y&8pfeWx*t}jphx-T1kT3Omd7N0CIAih~zdmKE<^r$+AKZW7%yw(`6(Qr4d>JMc-7ku)aGe)oDM84f_+h%Bv8PTyW95?cALnH`O<kHsSqz55cdBx}`5|Bta>7fX#;Q3|kp-?k#m(a4+)cm~jmbPK2!11Aq~t!lGLCrgljZJsrs<Htc^AdhVX-q4`wFp10>2dZ$0cfDyza`6t2|**SW>ketRUBS24Ss=8m`IS0wkh{E$&VLktD_+a=khgO%+^M5>BqbFB=zfmAu2HAmxU>UYeC+pGRLYUUJvGfJ6z!SfPlB^okNfs3pgh3q4Rf7mplIh&fUfz(+wh*ThTTWhxfT))8{<JgPkf&~WSJ;Zw3=9Bg*91UrMsT-s!n=PRlzAJieN@K%bx7F`SaZ6%dxm?4f@#MLx+Y9N{}8Vuutro)6*I=}urZ5aBG3!{i~K2g4R0veWangT>|Nl#7(04V~K<tof3`sA_AZ}g|Ggka8iRUU&dy9yXD!?NPOeX=jNNzTh5N6(G>tb0vH1y22z0OXhyD79=^mU-`5W5QW~5=Eg3PQd%x5Snr8L^0g6@|sg5%>ooof4<R{maRS=ttCgX#grJ60a_Fk8Pv}<$@D6dV%E}YDQ3u~qJp(wQ-Ro`sju7;*xnltoysZLSXQmTZzsud7@eTwhYh|IXAQbFtL1uJ6H*Eu@gZ2#Ae|wqDiS&LRat~lNn;RP@XaOo{GdF~sqd69wY4l)$wTyD5TTa7^hZP2&|J5rM~?#h=?o=Jgea{wGQ=8W3m{tniV!ArG)9dN?}cD6=debF9)1DgbnMy!ZZ82$x5wmtVza+z&4lbI;%lFAR<M#<@rO2GgRV-uk70{0fhs#xPz=u3nRV@@^DM8DP^J=V0wkVQMMMLDgRzhblco+Bwi2RNNIkeCqXiIvDqjbwR#aRlRY-voe)HN@wUY*(flIu&loWyGU{i6Oh<0XHA_Nf+Xn<IV(Tj>anh>Tf{?~cAK5t{h5^}+=C2FWOEUjpBQg((J(Fr}@HA4gVEkiT-f*86M6ch=-0HXu=kQpX^v)m63)NaOU#!TC0>w)ua6(+1aHvs67c@i!pL}A(V7wHmAm){S#&qrC?2B4|aY+$yHH&c-objv_H<I4_%va@L$+Dzo;V#cQC48t_%%e{l(@ymlpj#wtd(q=I19PmYb!t6|1UPR1I6_saWQ%t4~Kn-C*0DPC)21xm)>I%`(W<!a^42_QohkIcTnFE-$?97tc>o09zRjS#vecr4CUmVKO5`i@PNw_+WLsIM#ceCid6j(VEeU))2>zNlU0~^O=I&-1d?W#fx>O%+OZih&E@piu}(77&v-~memsAB*SOl}t767qpfk|2l*8XHxLaf@yUMKE`oIyI5&TuMSWB(?d&H5zNN4A;vOJuvwzt`oI}K);ygiqyPXtss9#`y^rn;rVy191nB-48Jy0$ZZ@)8GyvJwSjCJi`v&z(#MVyEI7dNZqb!U-%`VxbOR90Q0tTy0!sy08ONC&14;e=wZ}^NQD-%BBlfkqcZ%s8un&o*<W4ezE7~<2Rgy6rL1?Tc=cKQkXhvQaepD3{WM~@9*(-&C24G+y-6x|-EEI6a6?H}I*ZZeic&OTy)c&gDp@zIt`^X#ms9)@tW_3W4hAW89ka8#;(<U4aP62HtpdOTQoymaQ=|9&wS!U@XI1RHPuChpNnijU;3afvO&7X~p+LbC0`A{nMUO;syT|7B(Q+kHwBHwWucOrxf({?!6dkp6AD>2|o023)dcwg>IS!!>U;<<$I3wHHS;tgD+Rx_<-J1nymi-P0x5)?~Hw<NASA1$)bIv3(dETB<Wmd`dhTR6)4C8AP~BK{tE^zOTeoY7+r$CBvXe^ZiNz5N>_Wl^1z=<3anIr6nFJI>Glu1Lw={ZljqGE!c>{VSCCu|^YJz5O917&jwc00+25l+J*CXIF2&Uy=Cg%@3*92YJ5n)1Tjc4^Xe({zxurgL=7jVcObDJnA7iefRf_{PdTf|8w>B`-og9V=y(1?d<ZQx=}N&Oc<md?OfD3s;}L^g;OB)yL$7Fk|bgXE`%rK;2B5)lgp5ildCuXE6KMNT#pn;m<81I>5BaR<=)dLM}G(o_n#je9(k*{_tws9@>q6LUX!DSmWpgJh`%&rkf7kUVf3jOIaIntUiX0A+}tGM-(Bki);8LSPL_yHj<$u3UI<yyanF3?HMzjmjYGX8%~03(8nZodhnyC5+qhcg+?6X;urb3D636;|$Royl(RTpOzuqvV_<a;(B<Z6i(1r;ve0Fs7e4@y;x_a{+&<lC^LfILtlcB`^OHz54eBCuEUrRR4M%U<`^VR;x!T#al!Ql(wE$;ba>UiHk(NYwlc}okm2I@kvlt^6&uG0R57E-G@TTIBE2|3G(6IyWp-UJHruL3YozHCgWC;<G2NMzM$k}U#?!~V|pwn~T*abL8!@D9pziW4v#x^qXjv*5fta{RJx7l@xzyMHnh83$%^PBx!AlTJHj&I(APX&5b)?+qXj(ajm*HlU*8tSzd)Oi$A6EG6c_Uem_FN{y?LYc;H9W30s@ipN=&tUJWsNLCe;aiOz`+w@sx-rC>q8EW+y*Hbwfb{{>)=PU)W*VqH_nFykd-{d2oY`1c;ZnYc}?aLZz2~~DuGqCOKbdX-1=2Js4EWJ0l^sFQwk-O_i+TKy)7uIOuA#?%SrO<0SI>UW0d5;*1bPykjARc6#Lka|8rE1k;;6+QQNGvecR2nqub{It)SNF1Zdm3UvAif7ZA&H|_uiLIi)`1baz7B{ifYu-`n+BR*3-<syj85F9E!gOFVD*et2Ro2+G7dIn)7aC~FKRUoo9dUbP4s;<dv-hb?TX1G^)vvqdP|ui)h&ETpa~1o0)~zpzI+xO?R_eQ`f&fz0dN}pDhxi~JNnEsLNuBfg*Jhyby=grCOU+?blYXhvA0OFW7oN(c8sr4Kj$f3gwZ*g@ZFnYOv;Q@`JxCLQ#O#q7^I!lbnWhVxrb4y(p95(J?pGIJoxzFDA@aSA6_B!Z}B*EF(J3#0)uvUJl&pI<rLf3*tC(d$|)9!D-nmyt0d6zeyh}|w7cWJbTv4LS>boPZUd~Yfwa2a17iFhL{8v=ky&*PEdndlECh+?GE4;^LgEBktwdsnoGYjZ*H)qw5_>iYLe59#DS61iiiT;HLT5O^tLFedoMR#prosXiMI|0lfN?C(D3uRwR41c(GSfV-iA2e#Iro~fuR!bOEzx?P-AWZ~P@UiL?SbnGf4vc&Zgee6e~f{d8-^EMb7FUg;K9*6VT)zilC(y+{q_l_+z;lRoj>a#NDb3|9&!tCW)RWA(26jO+G$hJ=XKXt+PhlcItUCXpbjk6Cw3R$`B06Cd`tu(x%TAsOcM-rFWW<Oi`b-n;rfnHJr+-T8dAs?K0Sx%I9Pq6n3-{$KNJ`09-Mx?)GV|gHT4|1);IxWQ}x4dsSXC@*to1xna!91di~92Z(Vw+!y;{9v1%E#rd`XR0ydKYeHss`U7Jc>eS=^<JEPb_>b{uK!l>I2R(l!~s<9`Ul^jj1Zg>XHc%7Pvs-@lMZZZy*LbZd5v{Dr7QXbABVT@zA2X}hP&0Ts<xip>#ussX+;OdSrt8PAP)2_9vC*qOphs-fxCgGql)}dV&WmRqtninH$P`-)5QQT>_B3nIL`;|iNLuWk(7(A0}r12NMNI2d8J_pYhS4mx9Z{2jD3h2njf2SRBqh|t=2UB&s+4#l=8Z+m@emWR9&>rvCuiJpFS=qjR&;2iZiy}~UOe((gp?XEf0}fL91qIse#m{yPEy4PgF7#&Z@-}X;{~R;v?f|ed@;+zQ&2alq8E}q`;cwbfHwugfBz>)SfD|7@ycIy*APoGap`e*SsOJ~@(YE&t>)m-6QM4&O=Aibzrr&n!R~(~3GoygsLL++1ZbGU+Yrzh=vXm}zSV=T=hS(IaUX^<|8sNimnURB27ty%6UB1i}P`mr(xP*YPT>M9sKhC+D=2HZReQ^1^#z!RGzy$gVuuzE0539JScF9X(vf|b+NDJvkYWD3F3&?}g^0}CCW6C3VAB$J=1E|?WP;)SLdf;AzP;GIsOA7gxcsRAp81U7f%A{LI8NItUH)XX<=2$c~I5n_dYTK>0s9+ZMh__rIKlQ5_|K8f25KL}DFlp>uyBySWQ>bU}QCNFjw_E3}n=-vr;Sj74;hx+hK5!Rz_0xl_uH)8~AzTJP&8ju5F(tgI{uER++p?xgg2q0D_g%}Z^zw5@ze>RWy`vw9m@}ZL2EuuuDq0<2eI;(ZlPBxkouZPsome2bkDX+lsfb~nu8i8*fS5H>0=qgcW;->9#@vc|!{&}g5Hke<Dr69dhItT(N_L<UQdC=B#&8V*7I6iW)P2LQ9_*tsj{Xk{A&dw',
 'crawler_snapshot.py': 'c-qZ<ZIdHMnc(;S73FqtC5_c;b<d2~ZjX$|;{hgYkG<nDcN;yTsFq5ta!V??s?;y^^aqw;7dU(Y2TQ=+ZmbPQECCB<cUf-naL0#n_<u|UW1CO+7w&n>tjww`Nj;dmbwE$4D)Y&=C!hCxa(#XM^YQa>^74&aUOx?w#v93WeB#aGagw*an~C@4JW5Zzakv+avz9kW2J~;9hVe9>?t4k<#nVA_4A1TL_4T!3noPVP7|!!~8U=wDPi9G)d*O7N<Y68s(`;=`y-W9JVVXr;r=N_+QJ+evO4pM4G>_7jH;9Jee4G#Bey$2n`|~u7rg@uN(sgc+(kL9fnk3`v$5DTtCr~}if_^fYjiWpo==yndoR8x@RZreb!fCi4rCe$d=20F`B2`BHhT+5CFC|mydNc{+aeJP}W8DoJ7?uz6BvAF#!1uot&(!lM%tp`+^^^aB7wt)uhfp!pUuNN800Yg`i#M`ls-7odKH|R2;K>-)tN1~l({MUSCi<7|PBy}>>t9L#Aj<VGT(m5Y`?}feL_g*<*Rb;Xg)zwaH0}ek&=9C+^K^{kqM1?MgLiKv@w9~x*=!u=R?#%sOL8iostUL8)o0>y^eVm9xHLNn(=<7PrG^<Oy=>#+&+~{r_RAjuKA?C`AKxq{2i|3QG14>~#^Vq$O$L*X2dYc^Q63D(gSEBKU%&O*jaP1ZUC%%KGk@*%tN5fg*R3dv!n8lShI=eZH#_Yuf9=*Q&%bgzxcb@aH*YT}>#wb?!4B}|dB4%zS@Yn(G=i>7HA1$h$x%b$Bp_nH*^aYhn52Lgjiyu|#QQL$1}=yVMNqJvjl%8APjZ3wC_0u3sy<9W6kyo~;+TLqit`a68NX~NvuN7z(>=e56FTHVj<yJxG%##=KDX_=+70t$68D3>lRV0(m6m6W0bjt}6<yG8ab=M}Ob!Nd+Tb5q_jWpm{R{Le%Y)>AesYyOn#=$Y)x=Q+({K_I$TmKCPk2MWeX3EujmsVn0A{0k*3Kt0zW~!dN@Ljh++*r%@hOdF<FFq|UwMRF9QU(~FWGxThUXYOmTM2@lUde)axj)@hNL#k`f=R-OgPS9d`Oa~d3U?nj;6Q^8~!{WZan2TRg>c+9270u_&^O{fvjPyERH)s64@Y-Fa_N*P^>tXUc%PO>c`;wZQS<_V=Sp_Bu$uh|IXC+0GB=FGz_<HRHFp&28f3}o`xbW^r?-7Xhi{}4;VL2j-pg!+Y~p=IQ~-9z_AJ-d<5JuNLsrn=imIyHNVAW1f*JI|4s^>Z_yB}x9Xh<J&oGge6NxEyLWQ$&b-qJJH02cpeN9q=Ad-xJJZvPtuqFw8{^qMZfAKK&l*jdzGIv|gwy89pbj*fs*H?(2EBx>(g5bH)-RdoYKP(5HvHzU#6z!vA2&T8_+~tWUE9VLg|X5G%M`hyeupv8{m+Nvd2}5JZ`$x*ndd;3y>N`Eb%Lb93q1ws85rRl83j#hf0QKfBlM=xk+-n{jkaa-vSgn2BddS>N(jLa_w<|XSDDHq=V+MUVF<HfJL`{thX>3M5H~XKv97o4#{-;Cz#6}Yy#Nsg=tDa{nMD~8*dPXx0fdD{8Nx7*#)AgdZc^cevvB=*24YVH?1Z6t@c^WyfrrZi@)b0WqS1UZ&D4<NLB@mPEgA0VF|}0DF7>_F<bTvip<;VFACEyA0>KJLH0!tN8AvfO`Ds6IK$&J61g6;upbbpI80a~^TQr21<17<da4-kl>H}L+3v=TI-EV3MyDoBFgI^Eh6lihH$H%?bZ@uJ&^L&&5kv#F@%**D(Vcd^_z7bl_(gc_ZH2;iu2pbYvqBn#=o_Ko^e1{)V-X9^w=GLIfUI=e+H^9OH<xK;60mP}8Jz8^P&{z$Z0xD7AHWzJlV1*NytZ|GSt0*rsju;ce+<+K0g`E!&TDY)Hg-x)FCkPJq@fSvUKFdD6v$?tZg-zP{1tBy4t4#mUzcHnet0{*W+wo4J_G9UpOuhOK<Uf$xk(8xlMFuvZsf*3s?VhHDW({7KU9Q;kp6Gg8<w0iiNyA*k_91o-Hh<H*;%#+0o$4UW;#JrHlQ%MuSMOPZUx5r0Sd@Wg7U8F!pAeM*534AgeB8VK<~$s4AX5NJG*0@Yq_A9*Mz}4~cyCVc5C^ba#$Sg(ijW&IUcEU7{RSo$b(A)WhzHS02HP@)tu4K8w!99vdNhi$A22SeW=)tLm5qF|a_~Q@Qo%HOtEcixdEJPjtlFOX&xWa&Cxxxv6>Xw@i~bS98$|=fq(C@7OkkC9N($|*f%`-VcN9&lv|!QhY?f!F4+2<AZA;Lp8Vj~&%igzJ*2mH|JO3`h!PM`O8EB)#;MJt4qE@MKJc*IfTC-kWW+cZ54p)vUR96i@x5F6-;)BL1(*GSjRYcZyJR(Crw?JR<;mjTpX!#C|AXOiS0^}4C0RF!Z%*Pmgvsz~;)oOdJdc8~KdI0}C{bp?9t=;ux$2C%B4mk5DHR`F?xL=5rG@mUGHf(nsPWA?&m+p9}a8QPxhiQ(KJ3$ozB>ynaQ^I-ZPdo_dJ0QqSP`?0@t3WbS-?ivOwUDFHYh<<!YC*kOMABXZp&=duk4u0OZEVDY4TgsMTz&$iMs(aC&$IY2YHHzO=;?Nm17S-=Ls>LtVWdvJ09FfI5;$Z-grjG>ev2W(z}3quK2KJ&XKk~hneu9#Ubv8&OG|=$s@U*Zu~w>si`UE8EZh{$ZVjb#G1M1G8cP9JNGv@yAe|A>M-@`!wv7$qL8lu0S=->ciZ7h<*mk^5iJUW`KC>&d=Q9*!8j3o?=wxns#$0fRno3AtRnuiPO$2uaLSp+ESB+sZmnzg!RhY0KV3CU}L7h|GcORz)mJcYmGGBqfISF{ia3f4;c|q)i2TRtJ&*8Cr4v+26MH^bg1k!+wb)YJ-Wq8JQ(#)3LijV={_`v|gLI?^dfnCy+PFiY;N@Zx77}#0eblFOgHBo)oTYx-4m?K?7!w#Z9%vToS8~^H!&(72NZLqN(Zz{78?kq4<4&!|+(NEIZJi~Xyq3{(MWqo6Wyk77Lq!`)4u&J|Td>8@KYv~0Q>&uQ|;}Ipv1cuW9ax}ICOQkl5?<6V2KRiD|;fiNYlVG!higp4tAym{^qPB`|2AYV`w`7L>21x^cX{m!=F=Y@e5JWEb-#J#-RxoB~5mK870P|3y{G`GL>3phae?fUb&pwQkd1h&@x_r~Ho53rj$GbgC`axr-rU5oVOU6XMRRf#Y55N}KEbu&BKZUAHEYE1H>n^1QU^sO6LMfPnwm6S$I!cW|%Y|(Tpft<Sxfpi<sj}`3iuCFutK8}<{aQg2@^6vJ;KmD^s5DOEjC8BEU(*LP;eo9(JPOb#g%(R-p4**{ccp8UK*bEvG^g#>LDB1?OQ``!_Pndn=q7rVI(rFNKDy))Xb&PZRHKQ%0X3NWveB&Rl&Zu|iXe!;nav57_uRr(-OS+#a7Wb69ZSc3=rP!l(y-61w{+9wMFDdHT0Q(;@qQRjfetV?ug)9KwXi$@zpMxn6)*-+(Q4V@INT3Vb%RPEL{QCYDWEa1xG)gj{3|RK*EpwD*-{e7&jK3A>;!2KE^<qtE*mjZfdKW&F4A&^WEM3C%UzoE0ey(Cbcw4}0aw|SK^<`0qy0;*oU32h6Qy66o4F|KY61ycg&%4PJD&=Zat_Rw;WNMIA3}3yU?j^;GxBL2Bd8M+poh{n+BMJM)eyMNNY-#|tv?R4%p+F}@0IJv{b+^;0U?ZNH4T7)#d#2bDl{IFJPgDF{gyd+2D@MqwnS_&7g7Nffa#(5IG>?z+1BtF0+L29S@h^LXx56-+CIQ&umj`pq`TDy8436j_#XgQFSfi3XrBkwAq(_dWKXRsv5;c0uct7n0s5E#aChr6f|QS}CdikI9r#iU2aMP0Nw7B`0M2)x?ifIZFibT5=lPgmlxK{FLr*j&c~;$5Tojj3MqP~922>h2)YOm28A@M87h`dz^wCr`kl;03@S931irM1o?1N>$aueusmB}0oR(og?pCwtGgLI?}$R^D;TVv6Yl<N{O#D7P(T29_k{O7xo1GlL2!>aI1d@6vMZ&!y@pqmM)c)Kbj13e2s#>+EHI0MzR<&mL5bk@=$pj~T{Iqb!Tt(dBV)ryQgVlJ+JZor}=#Y)1wsf%GW0Qe_{NDrhWed8X1bkkYMtb7W?IMay6&`3s77*~gBvPytzjT%gFEvg%qwOG~{WA@P^<tkn+X}7UI4@qJzyt*$z^h2Z66EF%NWYYE&<Q~`{c2CR#{0ev3S<Cppb8KKjh|?1)TWK*ITm7Op=%iEB9@`86*M!=q(LS20Psl7{;1yY_i?NodjB+J%UC^X*vDG#RgUi!mG@pT(0W6SOH<}ae0c1#0gNnhBf<{&R64n=G+ERi{r5u0JeofkGE~x`?1K|J+Q%Ybu^<M{dbK~khh-a{2U%v_+xY|d?O^PzC;^ruyj9VbbkTC&3Y|aM5XWr~~+E2GGY+j(J%l=uj9Kld+n`h-Y3YF=Dia1IJdY+U^s$n~GI6R0g%!O;(oJKCee#J-_h{w7J>ipy^@|%TMl1iJdZFvQ<Qd)X71M%S~sUQOp07K64V|JU6*zW2ZMZc(?k^JC21q}VtrHjq;fQHdPWh83@+>Hp;Xzp?EDlE~7mnX`-g%l_{#Go`K<U^bqzF<$EM{BbQg5e>$0zpchfiFB4H;tmuNPvkpQ?wK6c^F+4PlrjPG?y)FxKOnL1FWu4x2qO~Wre3Y%^G*0isNd>vp^mj*qz-=z1>a^BX@%Rak3YV>xi^Dk@IO54kHD}Z<R5@*s`*xf&|$IQw=hxlTluYioA>$IKmC(IesyK-37Lvs9UmP&}Oi3y!eFNqN}2#*!78oILJWvndajYmasi>s@av1x}kKe$gie!nivCg3nx_%mBeOG9@c7y{Ws@;F9t1%#<D2ZcQtghL3WC<dmA(<?MVVP5(otWg18jXVO3^;+z{&|P!Qd3@k)5ira)~Rg}`Ki9vp?hSHZhxSctJ_#&!TQZ8wQkV5X$6CXXWH1<n)MEF7ILW2ZTV+Kz|2joNs&yJ#j>!PWE4#omdUf<A~qgXCyhvPDoAvT+p68rZ?o$MQUBHx9DZQs*5?E^n1WkWo*#|DS4smB?+8am-+i1qNeADS+0RR_dX%0$Ht89k&Sg3C+C2<Wn;zV2F<zXvh~~wCLcbx6KYVEd-aUJM0o#Qc<!+lQ5VFhuV@QC=^T)g%h#Ap#T0zn|?ACLIw^TH39BP>yPGBBq}+W^v7gsg)QSOIiy%5C@oG!0?J-$F?}_jynOlMlg;|o;YOa|$QpvxtSyU*B^7^mk?XX6-4o!7=#fIF+|Yj+qhaz<I0b(eLOb&ciz96~Fg5y%J5$;?3S*&vmoC%ZsHGv5ss=r2o_AbTe6H^{FRn6JEtv2{6mU`R|707{6X{zcTFoexwMj(amNF`9{Nw@w#xAxB(HgeSyic+7?Q7t6x&bAptk=B|li^fq3s{VxP+^-qr|#Y;YAsC3G}+KUh$fxd6%fN1N|cwjpKf`VI~NyV2U8bgq)H1QyZtTH1vjn^VRzy*cNQRtlU*zrmHH{Fx9HgQ@FTJr8tV1p)|Pf>VV9g|52TB2e1=J$%o<uIpwA0BZw#POjG*P+2Ah{Wz5;{>X5xr`HkbDuRS-8W8v@KO3NJkezlNY-HQE|h#hRLHU#!_-XyBF@yCelyH#9oC1gw^;mkTyPG2PBikU*M(B^!38t@eY#gv|%a&YQ;j@ze|!!LO)Up>u;iVi5HvS8C_SxxZ!!M5Kq-ibR?uc6I3|Y7I2Sm{1b~dp!ljage`%%AGmeIn`JfOu-0t=Bse-W^;FE>&e3N0<HiEW2fleQ=e;t_RbGFC`b<%-cYk;ue$0Bx_sr)ZT0gNX@rs}0{CJGWv9PH8qjbtBnEzyd{LNgW(k-QV3i}>1kh(zos<O;DCQJ}Sks=skf10*vJ_Nlo8$}%vf667ZO2b|TUAB#eEdqyvofC00J{DyHf2#it5TwjGdO^tt*&okL9j@aG)Ik6Th${9T<>7b)xF))g7ga8{aRfU_QYC)oh7q|UyQ*D&oZnJGJZmS2vEmUPm$mt2xemL1AlW1J--;@fE?SlPk;;(!FIOeYWet@vRonpFIQkV0(Dj(MV+7t?Gx-%*(B0hfzuc~!B2$K+)oJ9m{IW6j0&nfbXL=ZS{1!*dP>To=G_csFWRTj%B4EkQa)HG_V6w-jU9UQTb=Do-Ua%nwUkFK{ZA<+4%4?`G0{S9ZG$oZ)9krKHd7*5y}W@7ptiFSztB~aW;*J-meh2Ozic3PN3qFot~ga%wiR&Q(}NxFko_Ux4MC(;4T<{$Db<0Br-kp2x+D!1cd(I5r&S!cuPF9LsKh|rqVUu_TX9&3RT56oH6t#X3&j2-<&PBNzX`9vSi7m1A(M}S?O|Mwf2z3CSrDGe<~$41Q`nAY?fozLwe4BwtCX`8V;V5NfFetDTnR^OOb|=Eo#qn!F{2@f{ihU{f)RP(06MM{8AVjoZgSMW;oZ(^$jZD2+k!eE4_xkadM$wozyfl9cgfv~1;A5|X3ijJF4n@_+I*e(SDFTp+GZGkkb$~9g&2e$QSCVpI)_3`1P)`x7g9|%+|stxaWY*^v#gye9V&&{teqE+1&r~Ef^}rIyOq^nMP0kt0SV<|XB+=q#6N$Af1cu~DvQFrq&DXQMWJAiazwZw;sR^8fYV!`{ktO7>P3zE$oCq3-j?l9nO8!Suzrr}NBoTjPJsw!QDV+yGHasCjFW=6ZWJw69DyV{N5>2O+Qss(pv%iw#{farhR|1io`KFTUU>{+stBuUN-f$eN**v|D%>gB*+Z?L$5>LC+|S*9`6Uls;D7>NqM|iOr)k!lDnOAOI#D_|-q1qVqI!H*y-}Q*L_VI?(mg!^Sb>I)4-Wh)**Q9wz-hMCY3}^#hOq0{BGrDt2EJ5lZ>e41AqPmFGN?2Ed|L-##$uloS^9IgZ@=nkXwCENPebsx1_l;;uqF|TUoP#nCp(@0m%}ZPdG+zKiNzcoYnD~rrbfSUsy64=g=&<uRk^*=is7RQf3pK#;6uxpvZjQP*Qw9$s2Kv%&~mhxR(0>dQuP``Z!SlnMk(ZmU~AA?B!M7>*Ir_v%Na2;!$vL4a(A06O2kOI%_wE%m3QfAyBk}*r4+1$^Hp)nO-&Z9Cb{j4wMTz>)f!$yda661x}TiJgvE@h#%Qs&1cH`^3klWb=r6i!m~*;arxj;sg`HM%sB3v>kyBj?*~`5b_79RF9*p1iE?n>$s&5-$Ndj|{9PET-2be$vy~^$F%gs8cvjF=eUC)o?VK%WF7ZU8Fus)G2?lrLxZ>r(443-&Q%Cu_W^3P?DvUip02M)%eE8y#jUQp}AE-MNeo4Wc5)FN9k>DDSGV!FlV>$Qb`2zIJq6zAeNEk{0S`=)eFEPzIY2}vYetcWl%!=7p*p2Sd%?DcdeF*Rp!79M^~qa4(EgVCxei3LWa!kKJGkJtfw71O!0ya$5dj+vCk*D4b?D9r74%%|Idc*PM3NZUr4w>gM&I&8XV3%6L~w_upN45(Lx(G;Eo8Tgu8W~{8R*L4AQabhmc5n9&fB#U{?X_OLSQxi}_*Je7l$*m$$G^lpGR3kwD7)n(i#A(d7!w+Nzp1LWC>LBkyDDN^hVCOHMM<8~f7)p9-hTSrraeBirCJDpELn4hjD;Bt!B>zi+Ld~^L2Ehzp5G4{QAQWVacSWn%!X?6}-{UmaMwBuYGul&+Yh^@V68t4%i>Bmc{7?dU#8PNEa#pmqemD)t9M~{Ygt;<;)azAoM&U4|TUn%*oseDK6xI#srP&zmwT^KDo%X2IQPy>QYgICbLsgIz;Df4Cl5|vGaKDO=luFCGkrC6VTwzI4N3W1(MLc;%lp~&2GXqU515@#K#EBnDhb@D`<MLE#0h}%hH53&KULd#Pj!wZ$bg5!B>t!Az4S^M8Z6hz9u{v{*`AYP$vpWjaE&LTXY!f9}<ke;nEHi9=qqugBAq!VySJ9N7Ym?QRZ>yF0ajonstM#ESaGsvud$!0?rn<VShpg7&^+?Vxadjz=7o%l!ZGolRR3dxT<7wq+kQMaoh1P#%Uw3F#7t30efstips$Pq#CzF7m5CeY@D0*z`a}c~pi9L_|YjvJKk=jXxk`B}bMAev}%jP$>Qen(f2mfqS^b`KM#M-kO_LWse{kZp10x~jtyL7k=2%``Ks1zj(st+NxD|p7}yeB)Y=hOBoV%nqYpXT__A&Nne;gjc!dc_bZvcH3hq?JPoxYL1QS}UZ4x2%VXCfNKFE`3$0BucT!W9x7Y6_KvQw>~ki?jP^Tb+}}KTkVr;Hq7!3NFy8%+3s8TYMW;jUFQ?`FX?07kLFgQc`lMGA(627L!Q7-ZUbG-Oh|WR7!FQcNFiKl)l3T{{8-^^u2@rv>Yy#)>ol;*MACSUfn`rsic84d{30N=I%GqhCY;$lI*PG}@^hHR5$8Tbl>ows!KM)R$I%8P7Mp}XI!4hvrDVaB30g1-PLmY^6+xrZaXuYm8pMZ$0x_Lw+M>VP^!FnDy+nU6)88lYZ!SLkzVhx9L`=pK`DcarcL<iq9PGCPQ{;Qx8y`<Fo8kn=oxv%!Lp(nVg43AHbbz4Gc&}h`!3gJl5MvvmQ2}Wu3h6R#PNun5pY9LaZv#}$`R6)&SDfnz_l5nbO3g~5F^ubiVzj%mJ|ykTX)f}UF>SDTn1WJ_Pt(Nsd`P<vO?m#}E)?Kpm0-J5x4@XTm-^+pxUgy`6uC_GMBpG`dKT5vg_>)2e0_|`m8S&6uFuO#1uCS1-nQ|yyyGm2)oxMJJ-0cLCCmu2aLnFa&u`Yw-ty^MD@LTYqM6{;NH8l=*9=4_y9K-E2rQYwY`&+CFjBN{1CB&Q#jd`EzEX@?Q?y2~*Uvb4f)KtIq*g^q0U=~8{XqCB<|x9PJGk@prWOFWjj^fr@r*)UHa9QqJ{J$V>x%M2lt{kbEAo!vfd^HW5|(UL(nF&~V$F_;QZ?XgoNB49o65l_8*R`y$}OoFsU;1GOWpTmk`)_(*6>&dQA$zMd1@$8FEHZ)kPS%Nd2|2%+b7=p-+uoq)BQWyg&p{R<Ido8tF?W$Y5aTXtl9YVV|NBkE0?cyl>jg8sHqy1^M3>3m{Axvf!f888UIs*%}-4>KlK-ogqG04&I!IGD&1_|MmJfH0J%?<hun%LAVzw0%srjZ83sZk%tr?t;wVPZDw715M!+N_ekL0YJ`-70$R_FfXD2!dKX57hZK5I_qT<)~UK-N**a4kDq8xNmf!1~qWqsI=;S`e`RSz+*NXRQ3nj(}UR{Qe?*Cn}xKJb}cOcKqSchj_K@O}eq#I3Wzxj%vcGMGdgnDA^Lrf@@c7X@fYQ&q|<WLs@@;^aOpfmyF)_@*l!pmhSFw;FGV&C#V@Y#GEKOp`l>?ljiD^;SWqHnJHew|<Ubg8>k}(q94#&%KgB4Q`%;+SbBHY>emuHn#g5oLwK5+bm`n(<(Hm3!+<1Pa2vAP?yc^uX1FAd`|gUXmW4gxaqxg<Nh~pdVl`<)tmk~f)^1hd0G%HY1(be8F$lO9-5~?3i;5G=t3WqJaYh;pAm(pR51-xI)SFp2uQmM3h?%8woN`G#Z{93q)(}K_~@nD?8H(`^P}P`#Xgskn}7Zvu6nX|WT!CpIP@S+O#ySoO)iS3eM%UW1AI>$duzKBW(OGNh@AWA=%{VHG4?ANDd>ig8rB6F2s2QX)_Z=%GOO#_%d!aMqv7x#B)~tws-;a6PO~FsiJw?I(+sNyj>d#D>Cu(BlZ!){(}5IODH<9G9&axVhf3g>c6D)*+AYd0Bk$GL6;w|d=XDJ!fT9^Z^y*f0`KXAjEI=Aqu+ME-NFz{AITmckl;$W+KNFf=)de_mZ+001G^*;kPrcy0w>(qaEu2-gUi^mW)0V`~_zp;tybTOrkPL&jO7Lcgn?5)sB~0>isqO`!_oga!mU`*xqSTYIsB<$U_!+}nP?JJZRcn~Annr~JQ|<aT>j#Kk=X78R0o?T;mmLnCg|^;kIO4Xw9UwVkmCW>~2Vh1v$&B&FATuzCxE@_V<hTv}v(RNRf{nK5yH!IkVawAaM9G1Bq@^^yq%d@8*sp8^=`Zp@`>G7O{e(}xpxO^z0=H7MSyqf#P^oqwwEIV-O_KJkDm5sHnOk&GrMf+AEsWH+bc#Q=w|z%<^}4zm9Butk?S=KzqsRcxj!F^Zpc<=BoW?c-#Sj$~ewBTx8XtGj#M74LeAX*|^u_#Bd>d`ab7*5I4u}_SRVL)n3K)RJfF4OVIKO^SYsGtHJwfFUSK>XU;!P7%u=iwW3=i)iA&K>kgIP@gHXhx#G=~}M3^r3XNkPN^=)eEl!>|6!hu`?|AHMvZ|MT1bhU+P>zW3K3{p@Xg)oi-rEX$JYS}D9;H!9<+h+;0oQWXozEExZ3{i(R9L5`iQOtZ1onU#L7=a%sVE~_KPrd*Z9s%L4Rk{E~M+<K3{YmW3)wG=IZ8CwGoYO?P)g+W#g1z?Yw;2GR{)!}uz%IDRWZ@jQkC=l1D`>+c~UbvUhbhNak`PKQ}%TQn%9>)8esVhZi4tk3Vqm<kOhPD~F{6$@=vf0?(+ISlBD#4!%&CO5yEeD6w3I%7<%11S35M^9eS)dz~V5um~Hw&O~yO8!I`JR;u#bA;{;F1V#_c?^sT|dQt&enUI&nY$ESSp6!2D;maL|I@KtcaWFNo&v=aoMIuY58rTG>>RDp<~*5P<}t=iy8y`QrOg}++%V<+CCU-W)0JKUNKjZChmmGJJ6bSdbLv(hfWu01-7j5$jo%aNj5gL>A;+F?uc{B)c}h*7qYRCj4T**c?Q)+SRhF5Z>T~QJFBX^%&e#iT69-c*8Ea>tUlC(8zixN(cy!y?&eEHYs2y5K?tM-aAOf-*jaH~71Xd2=H)tx%g}$}l^Z*V9Pyb66k_so{J!g7m5#!GSAFhT*7Z}(2c#E^6*U51R?2KON|8Nj&QPYvK%eQn1g^;ea}P88Y8qajFTZkE<ZoGRUFj6<<Q>|=E&UQ#WQP_^)wRk8;0<6O?GDA(mmt?MIW>MGf4-g)ce%CIG&drXL?tZ()x#Nfe4}NhkB^)C1ob**Q+5J#({pZeE5JiKZD&Y6d;X_hec>}3BA}HLNU{`W0IMFH)Ax@~z}r8xLzd+wU!y6xt+<1jF`V4`KDI((HSl0dK{J#FaAy@rjIZdbO4`_X-3mu&Fe$~abOX*{*mKEK#V)~4>Dv!Tju0x&QtI;kvma&6nwW#qKH@2$jRUnTBLaiLU;+raI0<GV{U|%Zj8pUT&}dv#UTg9`pMR9YF|Ji&m$y17DFtx>c6YX|!;(@?9hdZ$RoGcun8Ok5%5)P>V4du>(NbMl_9{s~WY2{&pyLn&d1z-vaA~5xuNVB*piQx#9Mkz@E-4VY;GePg+2^+}Zf*1G10G#FNn^Ngx6fu_0VNkw(mk{wNXc99Zgt`RyLZ;__`OdzcJCZ*^qy$?<^2OSZ&-{&s_&SAmXFX1Pwdu{7oWP+(+6Pbx=?a}?*sBhe$#AU-Bk(=32-gD6yYjE-5ui6nweqnqpD8^p#43ZIBaWw?*RWf-oOWAb&d9*%+^z#iyh|qh#;V-&F1O=uZWX428c;cB^O|Erq5`6r#}5-1+1DDRhJ?JQizzgY)2fv_d4Ha`U2IEOinK)$-#W)%Hve&t3GiR8#k}A_DF1`J?-zs?df>Zp2i?DCWl500a^UXeWC8bhJn-%Q`{05foy9RSdFgR<LfdT#e6hNVb87%R#ZD8;ffE{y2f3!ZK;>Bc9uAc>+39Jiw64E++t<y8SUCXn=XPZ!0p%?<8tb@;%db_&@^-@20Eb&R|EC|cYV2qfi62%w7pwp_0Iq3zMcV6XO4|2E*|FhbZeNp7XuEyt2>*_qeXm?h;iGn1URq@j0Q*hQL<xm?<sajaixOU04$3wHC-I3D1=`Yi*v@LI%GHvi6ys|A*Jb0@}}HM1tHLSP;6Ek2@sz3;_Oj39Aq+@!Ue{KzKvmv=w_tTiF+AFhLlxs&izgnVoVenQOp-5(JmZ*Mf?-5kGiNV;CtKQ<jr`DC-5k|*G(~&yYO1x%C`5#gr(TaHIB~J=}F!G7mlaMfQgi@gicgyS4#BwJ{p=vd1K$~MBfx?k;-N|<MKhEM_v;O`r?QWM5OlN^c=)HB6n^5Ojc0F+FjVG#j#CQN6Rkw(oyPTmpT!2HdSGOArTZEj|x$du*E4jiG+0s7+GX?HB8ZHWj$|Wz;qxo=bR1MHt%l5=3Usgok2McTQ~kEcbe8)SM?xKcrk=`BVx2^44mhq<nLe$F5hi9RQH|$M5u)?O+AV}$SHp=^n^4^nG;LtplsB~cDkjj+FOMYwySWLweW3!5l5?NvR#~g>;QmNhF0vK@s<0&2&zku?Xowdj2z+!aUp1Lg^sfkxT{UT3e9W-uzJhpUt<|vrrl49q=?kQRaAtD>w2yoivn4<z>xvVdTm}NMxG08(sh*}6EE`^oas%u=dDu5EUm9wkQ$bqlZ(q8WC0GxIM-7|UE@Vo(UD&Y?6E5sCv7lTOE_yK(zdP+CNQ8jWM<84(ROrjV@q2q;l0XD)AC9J6CDXyQ`e_uMZ`9e4DGyN1@R3^%?~=ae`kL2Qm5l%LX|%Lffr8?hhPMuI9`=PNEzNJJ<&wM-7)-!RC|O1bq16l4bK5Wn7<rN28&0|5ifKFcpHy2&PJi!-CMw~R>V9#IEG9`Zc&X))v&Vp=$m3yS<05VEys_b+i+7S0^5Q!DO4`PYpkba>D3M62lW!+Q3r^$bwq7{jY6`$5#B}N!3~oq&wvUaXDZb+#i7T0DaEIaL9m&lc22AUt#D~u?GT)PW4z!vZEqJCqie!qd}$P5Ypj1#evU%d_IKa^%Kaa{|Lyxfz5lBZ-ud8{@BiifzkL5&_rG!fyC1y$!Mi;brRoNJ7+Zi%3Skxl=@JM8WZbZFw?Mx5cmmgF5;4VT-R{tg>54|rc?K^^iXQZNYP2ncbyEVam|w+<DV3b&O`C&R<=qBMI-dk6@>It<TyPaG_)AjC*oHUGfn=>NEK%Y#EH!1qdPZWc;U*Bch(z5ruUuN-5Sq?QXA3HmW=x5D=B#m)C#w?jvvdkdkw9t+wmZuw&x{39^|qWv4P&w@c&R+c#pL2-dl+sgI|z;-!j&(042{mUp@5<bSCo|n6FLPWqV?Gje33S4QcpbySeIPj*Rd;sDGuTW_RCH%lRCnV?sT^50Ej-A%`?1zx`0rqq_zf0EK;a-M^o#Okn7GRO|C~ydWyZN?TgY|ag+0r6PxV;)dM++Lb^rSgmXd%NYax%I+;l4JkXMvO_GC1ZnJIiRn2%Gig<wG;A%uEKg{4!rX~UpvNic1{`%_=fA-CXzxmz6Z-3+AFMjm<-@c7(?crB{{rh|0`|ulo^YOc1|L9wP_0f0#iQazm-bdg4`-gw~laJo}-NUcG_3^KM^YFj^<l}dLh~GZ^{(t%K`+xuOTkk&n&X*~w;eUPmqo4iY;k&>6_-j9{N|OHXfAh<SZ+-9K4}S9S7vCq)|KqKPU;Fh(|NQ-rfAy<}fA!UezxWNleg9i{Lcp7j9=!MS8Jrh*aPKSL-u-`^!hJ<^ctvNKu8r>h9A0Gi-+{&0d+^?m5^n<Ej77$E<RPoYqO~ht$$Ze>xmdSsdkHubQd*EI9x0}K#cN->Tn7Q~+2Deh10YT;L?(V|vn9JWmjOadY|??krg<MJK^-9Q69iK{z8NI_)J8pPN4Q1RTCjEHtNnZwCbWR5Tmhc4nA&JR*7D9&h&*+JL%|l1=-OEUK(ZgD!je9a0Bz}E<P9xpX3dp20bG!IdD+YuDzGhemw`*vOxet9!K5-o+XFo04kEo9$4u|V%W)qR;$)cLz^l>Wfcf>~*%&+$>D@_|Y|uG9ddm6n%42tSp{;m$_X@pu_U;uhsXQy9mpC!%#+=YGMWnm9<L5KjjiJI&PIETHlgTMvrssd5@#!5%GkF)XJKjY#1Fnzdyo-0=Knq{!O%mihcd^QpJVr9;!Cg2xet<R>bq!=RMawR{$CTzsl22e#;XE^3h7JS$!#n@_hrj#b@Bi(shd=*55LfyA;otuB!>|AEhhO{U-5m2Ucn72Vze1+);NEwDh&_1ko%pVLb^jkHzz-h0_u~_f$~OJ#Ok3j<^+w6=GHo2t255s$I8H)hEEI`;C)jg3(m9&L#l$rO;^BI4gohzBI07b<a`z8E{pQE-e)*%fzkIhJ&LH9_v=^^-^>_sEKd3SWiDh3&0{ULyLL1ZuMbgV8C!*+mrI<LfC^=x`R;21XYj!>f)*;{E>yqv8h?2Uwqcc#HG~XSDj22qhuG(8WEm^&lkjm2Oju!WeytRmc!c2BdaR`qGo1A7xai~%Y7E>_+Av+RpfJ%0}qY`nGAR!5iB$IBlsUie8GY<HCAk`{l(&o!0QgWln%IYicPcK}cGQK2Qvu~g%-z%Z#)JJqcA&t%Y3v=UJJ~7GuAzf;7wj1bym@%r?cVjMRy4k&JT^nzGmk>HhTrL^QRYmY;_b~U92879;AS=0Xpu<c<t}~Dm)K$}vmo7cm_t<w;t45M)^<d;9=daX(u2@h{swtKv+p@B&*eOpO$(O4Rd#Yk44spUVT4qm|9dV-SlyZ4{E*n<O1fij)=96g)GB4A-rBX()j+B7xoze-1%@MEwiU>e~%46-AE$@`3MV+ESZEFl7ecK{!6@y;IUS^uDWFi#JsiZxs<czV$CW$eMWTKKyOJ|L@PPf))rgcJvXbi}dMieoya&>hf+fg+S>KTNg!HY9r@?<yHs*v&KWN~hEg(R$;xKU@5XTU~oHkFo*nxedH!}U`l0TAoeJEKHw>oy_K_x<84lgNQDU_TR@Hcs{fzP_YLF@>KgQ-^%=G4)1A3qdq3`eXUV^8%oC&3W~T^P=GL%gy-)LLybzxwgJi5LkIU#M{)md{>e=SL&N2sZ-s@&?=%XsvN7KR0?jsMf<a=xhvmv*8D}|d5wyCL@j|i0w*4*?n{!kWTuV|;$7Q2rTL~inB_v|W^u2$yg{aZx>mlY+_=)+zJ92BN&D92cBfT$>H5VEe5^n(-KYL^$An;Ck&Xb)RC*X@Z^ErG%7vdAbW;kgpJYs<XMO*?hg{X1C|R-r3aL3=obA&MhY8*b_4(_!Uc2$iP2{g^iqOoySl}R_=b%ZRc4!hac@^zf1MQ9ZEcMODo0WsGxL|tCeRsSrVrBBXa6$2I>tc8Zkl6|7T+^T!XLgdc<(Kju@_~O;=qb+!mr*}mFhQM_*tyf$j(Bmw`dozgNdp*Cr=#f_g&>2iorHkqN1T+!j+KxIzUJAP8JaJfi=`?aE^kB~VUFB9I)?DIOd{-x9ZYfhxD(alFWPv-N#GPYH=_35Gv1ue9ZaJmOmgKh!@{ftzzXG<GUg<J`~_1y)uZ;RL33G_QC(15`G{N%MjQ)W_rs^W=`u-}Iwxgm^QI%$s)`O$QmEykU{{0)i-m6nTY%;%za8|#jIIM+hOWIX=6x}vShF&zsM>lpN>{drhZI9mEKgCTn!*VSb6dE6x?wU)@;4H4^hf$tcb~Ae%*7}*NdWUIo&9M0?vAh_K;LZA)HV@WA?cBE^-0YfEbS8B0wl&>volqPsw}Hg4q*-iu{aO+E&2y;!vvHVfu{_DRHTMxU__a~VhgsgBO_#9ql-N4?wAD`g(P)T%d_>pC@bD5hu}3b%BCp3xE|MZ5cBO3;`OVLVd465)Str^Yhat0ZV!`*Q77aF$1y`mLsOp9y;q3k_@SMFO%0R&C}j-OYL>Mzb0^4P?F62Xt|dBd32e%YKb=n^N?#fx?PWIsoGjqRVO-_BkO;ji0NNa#R=TuIrgE2%Sc$tmyy<tVlrg}bX{^9#S<<;VLPR;w7C>U-f|6qaQUxOG>I87N=Qv7e^%-}fA)~7Uq@p(!z)fw3)5UW9b*Ju#A?-6=ogF7S3P$|2ypR$oqbwK!hcegRO})VGv2-q~+nUk|HHe-M=95_liQXttB`WMdoU~ft)1eAY(2N@?wI060F}uPVU@bz6Z*<_-j`>~*bj65Mi)!Zrrz}4;Q`^FMT?ru1qydOuC@d&u;|x?Lh>{2ClA^sOiWG0(oo6FH(NM-|+Vb9~OnnA!P4^Trz+ZG`KxpYoXrKv2fUbLp3OZD|0(TUN@i7(dU_PdEGiC$lzy)=vfhn_JCK&LsLaiTV`K;e(j@bm`8OVjGjgTv(eG^n(K1GMu9%{V{6O!?H#p4_lB@irAQY}HqqD=dZ!e9ocHD7s+@^fRxr$bQeGBhlLmPGGRIOjYBF<#3+x8FjfDX<V=>UgGMeB#kXeki7q85^F#S4(S9J_<_8A#n$gemKoF-3H(noQfn)I&0&HPwerj8W0u-ru!CylIRtrs)LMQ<Xb6Oh~zs7if}xre9^Rgrr9-khP=+A=%CTDtW>7-POcnC;B#@i6c;3+my_(UpXSR`Qgb<)0E)(xK4IF{1o^mpf9b{G=JPLIdl4C+D>t>I^{vh7%OcS&kO@GQ6rt#<n3?)4mV1<kmSSH3L!JTS7%>LENIvw~Btik;6O_TLnRu%rl+4AjKh0OUMHP#|av*B*JhbH9ORrpeF}VJpAhVzG`{wiiZfBx0ypuQ@4;J4nU4wS#LA~?g)9W{#_~g8_V9C<Fep4ks#1eR1s%*XDSQ<rmG>I})qoRg@DyW<S@`)LjfJ)(vxQdG}Qc9SWSVt_Wl*cUN9)DuW<o_?b*HLEcuV^8n3sb0)eN~K8bHxg^s_U#qTT87am{rg|EfmxOK1m^lcFJE?R!?uSbLhqkn{YeyBE>W<E2TTGRN>;bAx8>^_=qb-d0K3pm2iPQF_40TRBJ9!UB88c7_MJS(sVu}bry_8GEZmgl|isLPo@a==yP{d6>(bevPq{*x7<q8iYbg3x9X-8kxXmT{7Ii$0o+SjTU1sPV+AVumlc-VTQlOi>p_{W4oi>X`gHn`GXjwrmo78G^fN`mN`j~BS&~R<s!sN7G2q?ul<LT2Svr}IaU62)ncdTpXLN+7(WDRe5R(*2$5I>cLxRT1{zZYb+@P5OoFd}~7<}Yy^_C6?I;4+<1*%251c2KV_&DJ5ph&}LRwjhK<3e1BiYdwl;OBHsO^92o=uop-8usmM4k#omuj{B6jw*mUV6TA~wMdsf528J0A({ieo~dF4vW8rE?<Bw&d7HL!924YVOlI>jyYi6I^7IPl0jDL@W4}GdvCDFr5${<&s&fmB>ik^TUC@)Q8xi(lwY6KWA%iILs|)xYJrchd3LZ+=KPV^om`wy2eBc$Tny6uEwm<_jb}64GuiwSSG%SQzMDsJiRVd7=unk3!$cM7QnY}hasl@t6QM^B5(uI9G-ZiJQ2iJtan*s{~?cleb)?OiiC}aa1uu7n&rB@uWL}(sbWAaf}Y;EYM5iQblkcp;gog_z(vAz_APYBg?tu!=@?^SGtRbFEFxlARLlPoOT$H%g0DpH_oerai2m%S|<Fiq6Lri&rAEG(g<7>dAD!AQu@a%_@<OZ%ncepUe~MVA8QYHEkCgq{e<0jX@PupkXf#0Y;4=~t~i%GHJylvM35!?GinT7^Nj=|Wp40#GQZW_8Th)QSrcuU1pIK$mO<CG6LT0z1y!4HYPA%Lf)XfTC%E@NHTLWfW7Jg)$?9(PJ?kXclP`svRJ{O_7{%#(WTmQwb%bBo*L{pmz<S5K4+5YOxy7+0&PKPO#1{U7FaB;&G$>^kpDubSy_;<KExut=Ix6K7d1k-gSzaJX2VsL>xd@A)2euaTsTGV+4h{^4vgGc{-~J;83{AYG_*y<qPFgVf42m06&=bV+=;*PQd|)2r_EGQ6dP+JZ6;(l2g-BId2BGCQlslnL2KQ_xn)$9Pucbr&Tg&f~j>bYy2W|tXY@a(g)jHTZ2ZXUowG%=Jx$+DY>XB&MwP+!|Z>l@KVmq4!~lSwEqzV808MtU>xcRb{!9)kCGVg-qJ-~SI6MkwQK`7>M(&hkkAhHu*Y=w1nQU=SfMn!B4)5+4z_O1F%@cVuVg2KZ^};uLN1+C#F(!x$BHOU3TA>V=mdEZY^yI|ik*`E!2VDxqu{;<du;%2w!H|qsrc6i0+8`1O64b@6vXVA>D5^J<zg)vOzCi+|G4+!gZF-iiJalA1Y36G9;eU8&&SEjH*V3nMld0*Z+O>65AMA+1s&GF%rg&E^Jieo+<%(_k$;@acS7T{XT1LXf5w@7aPRNngL3d`$pS18>!m5cmgquvkRr?~TC(T?)I7cab2#B~JU3b_SO^w)aPOb!jK~kB7`_0-W_)r)^<UF}cMq;Tc<&#k-Z3D;7^ADrjtE7=ZcG7w><#A7#@>T_{}~J<)xyH~Yeste!M*<pP2BryyOTgb(BuK93x7nDN7Jr>)-bl-PJ8=P?q+U|?*GdKXYDs~JA|4AVV*FS!w2vE3{=wi!M%5RA-z80#Eloc{Rj8H-LHW43F?oGd=}&TZx<IB!OD+5_+<#TS-74O<I#WT9RTdFU@!3PgM0q~xB(ICZSO^nT%SRqAH^_cE=KMUP-T3_a9#z_AG||#?!PlDW~gLrV+<o?kV%HaKtG$S=y^a7SS=7);0tCU0F>!N<S=v<K_(E?f=Df@?0T!3!qmU({l{pJvRmK9mHZn3ipzG_dzsp~>wT`+7g)r^TKIic&D>qwWp}-ejSY|f?kxVtFBDW$iZITQ;s)2b{?_k$r^f}MKJ)muskx|k1um&Kg>Il<83q~ks?d4h&ALHliiCocNbYf%1DJHW%HaMxu@D#ki(mYraBag`sL|aPiF++_6i)N*c9X06EoXLFkJ`Hil!y0L?u|eA<%4@an(nXGZ0UTuyIExoszz@6aHqvfH*Wt$aO?W3uiU!bo)BNsDIxf%bH%1tBjVuEoFNLqwv8MvC-osVj>0fX5%XdlSLI!yd~UOQRgxp0q6E5?YunLnRU=WYCCFv(aoK2`(jCObDfd9x!(6#%PuA1AXOFB!O1><6#>snYyK=RggMFr^rvqH3*V2eC>ZQ0i3HxJD6Nh;LS4DP>nN=~Gh~jRFYiRkZZ}hIq(aak!Xg@5!X;s`Qvbf--fsQ)7EHFlRc3ARVHg@F*R;RRNAd)Of$M`85vD1SiU0)?axG4HbIZGH|R_`;xxtJWlNH7a2<=QGd^pp=5yz+7>oY|#TjgQGV12mfF@wkoo(eZmRHiXy`O@MM7TT`b5NbU_TBL&`bZ48Y{#)o(vc&k7;EB0N#4Xilbt|Lmfj-Zt3i2AJ;%vdM6CfF6nXG{QPBCk|xdG_V!%6LxqYNJ(3jOsgupGYei4a0kv8Tlt_QD|ico(iX?1fHDRMARa41mZMnw)e-$Uc<j2%U?-lVtdAPI&2kM$YW(Iyd32qHv+kD%+x8aLo*I{7))pev_lZS?aGg`*d2y(m0eE{RP~|oewA;0po0u0uNjoKePnBHZyTK;GKad@hvo<f75AC+7uzjg5ut7*6-VCU@o?!j?>z@uK*vUmS3w^*qN2(ho=7`GtW1D)N@|e-bZ7NTtUs}1P}Tf6&LdIv#oWiETxoX}oLh1@Y5(1)FRB~n`O_IWM*k@3jh1!DmF}qKI`Rbv$m%g+>0z>RXmq#GimRvGY>r30Cazaxd0upYi^9>E`P#>}7KP!d2t2x(0E9Hf;5d5$U7#uTuBQ7W-*T$c23~Q3{1@cCZa@y$KzUvE&~-<p+Tj3Bsm=HJSB8}!b0}v01T6Gh`kc5yZD!<N+M}9W4H7lL7{XMYa-Pw*P`cCbH#V3&`ucEF7ZYoSn{+{6AVN3*fYT>=Eet%k<nB<lIY=6ZN3}7Zc8o`iRkL`8ueMS|_f=ObDQji5N{shjg7#xo8%5(;*S~>@QZqc+2KhL%ybz!Lg}{=T0~Q={6j&OHM>iGnz0I`{(+IqX8z8{>3eqin<Ehk0u#A)}0B+PEw#(<}H-rpAHqimbvdOiws0r$pAP7B>>B%6de%kK}2GpYnneRS{#g(H4?)9x*Dn1P+bszvOG;6hMkwTvjbG3G{cH&8#TwIwAqr5CH@H-4LhVVg18D%uQqzyUux1o;q)r|0R{SXBZic3qN;F-UrP(V7^kQ0P%C{HhkBA$&6JZgCUuE1fG4bTydS+7T1GtRIccbP&z3i4K4WLOK^VivQ6n99ChY0I3qgWLhphEh5mB<zCzLD*SCD`TZo#Kr#HqQovJ1C@%X%}510jsnKcVIbB`A=r_K<!@<hQ&rGMi`Bv6XFO^QWG2`rc)ie}qGDu}z!B?cQ5cIl_e~;A5Kx|=Ktvjhoz3zA>g&k9@$4ZWX;qn9Q`uM>&9(mv+YRe2',
 'diagnostics_snapshot.py': 'c-qZe>u==Nb^orv;zmZn;d(@^w%NoOJ%G*b8rjB=^?F^SQ3aYqa%O08$O#{k#_9kq{2}Ov0$sF05d=oPF$~v9gC=#0qN@-6ASeHa`j_-~&b^nsBxgplV;B{%EAl?hJ@=e*&+}57o16Eu&?`dWiG1qCabKjPQ5+?q2qJHsq<Inf;>V93i&2(NG04xn(^zC>(ic(U$7K*DV=?oxBJ#L7cQ!XSH=^k*%?jaVV|3-=22b|WI1c?HN|Riu0r%4~DZ;ETf^g)OaS=p*p(p$KiJTVzy&{^1dXj$c3;O?qGzs;TNimJ}bCgz3{40)z&NM7M%=UE8`+1tkHD+Eh!3e#?W4!7?v)C&}X*SjG`8?N;w7!1+AexP$INaFSz+MDs8n^_$N09sC)XNf3Dz^t40zXAImk;!lg+-YqjEobfUXb$~hd~tKS<wp<KP5WvTV*lYdd0GBo*SO|;j9q7H{X7kWhsdG=U!Zf{A>#^r~Y~ojez6LW?>TaM%Fp6l6cc_F%ah%ys+#xPUnmEe&XdDibf(y3nrrzo<(_)_w3FN50AJdOHXsLuis9iVj}0IK&-o0+DTqu{b>yk><~#g9g^g*tVW3l(GX@3jh-Cc7dr+93P&33pg)@RTAdxPm_LNECY-@0f@a&D8Hz78H#*EPf7UbRTyC?)Aw^F@OYsMX#U0LzFz-=gUyPzK4m5KlK|t~-b>7D2lkuZBdr5do`0rZ^=4lq;Bw`RiIS+6S-{ppinhcSU%J(JkKK4{(UheUc-nSxXi!5wFbNJT;vI9V}gIEU?iDQJRA)v+#^z_b;!}&lSilCg%^4<y26N@H!nT4*G`%$$2su$;>O`MJLD9H;i@k2cj?8ysg7Hq|~MTmiddzNyc5-Y6Qm*W(Q4)B>mnrRVqQmfqmqQMJECTAQLJh^8(EXiR{%JEhnjX@g8ET|3ez|G6))XV0*v;>+-dfGWu;YpYjbX1Jfhrwc7SaCYeEqaKOlhpTEsX9EuGQcSALp$?<Bf1;(SKzTP+9ojXS=@v(x4$neo@WU!3F>|v(To>UW7ry4N2Usp%XQBFxi$3iD7OZ}DiiO-i(+pWhd|cCq>!3_G`ai_*b@Kp3xVB<VsdryQ6$EZmk9sr;xmt+hr={I7SpSXUqrj&SWd{xxyU?GTwOf##h1^nE`CnWzZN+f#A%uxL#nJrO$ty#;&`~VqcE>%sQV5Ly1G%Ib~ww@aTeywXt{<%de#c|v@}#j>LvnA?z&0h19kB=_=;IAuHIlN0Hf7vw&tOq%H^q@NO@?5u{X=Zz*T*UYIO3J&Fewr#a3m_9kRnwo|38cik@PFS6D~3?G$O@#Tu^CBrwZiPTbT4CL{>aicNgx&n4^qbUKT}A`CD=&Xjwv+TCIj<!-%iFzU8CK>j2Q%D6MNYS&ts+^BtHrR{3Bz^ZO0+Ut%3aa*GK2Dvfjd8sH?9~`;3%ZJXu)IEjY<`NC*I5n9$FgV$-K{9o*XOMRw3{qqMC`zIn8$$;UTihe|+GuYxcL4b*VXxGCU;L?UEKrlxE8N%v)|xArSakwyO~gS>Xai80#kmWu<3eSYtSIENOzd0Yd$(y}mUU{VNNMg>BFI1ht%9%v{?Mdr0l;bO?0V{*^{{j;8e8?Q><aKs4Val;jgE5Gm9)^(CACg`YAUr!m^QQ`J^#*E$}#@KZ?7&sDU^RczIyRlB9h6~i%*MPsR<|P<;R#qEh4}~Xw2;LSBccM(bbFpN#r_?c=+_5@B#a8MSS_OAh^Hq75rrESQ5?JF@%5Y!yae;BsvLuFqUwKVb=OWH7R3;u;4NHA7&>YMuDU5>$y|tCUH7}^DN$tl7PI!VHSm>cn+jtXhA!{J|%@32t_!;(DvFZXvKncu$PR(-YbS<*Kz17>>RarPb1kFCePqT%b|N5W>8Q$PSR?x4(x_*08&{GeUR@{_*gU=N0WGLGv9%1?=Ix-4*o&-jBP;me!uG{W?qyx{T9}kc^rnb9u7wDwsZR~PQx8>>y`!u_2s-`YnNkKMsZLvi9L=cd(0*%F$#ca7*0MvHpoPZd%srPc<t#(%qa_Bd4S5w)t>&;OQMlf*>!lNtflT_BiVIfJUUuS9U<AZ!nTHIGX$KhxX)qjVAWyD2cl&!C5VO6zJ8jKHp{}Fl)y)#-`889S>&Dbw3Jk6Pa~pcTDCKe(_zoLCAVTX!XO&b*nTUR+A!pZ1Vzqh3Ir4pPJ-dbl8WJR%9@i2S5j6HfOcz)Aw6h|@9b9t4rQ;T5KK7@IBKqYO$7Pq!S*{(UK7a}dM1L4_TtxN<)Rgrzey(I<nrURscI~H{mw|GBD^5>aWr6T0q?eU*!es)AkjAZY&_^!kWRgbq>79i{0P-tM1}M~VE~IVr>99#%-s6*C#4rh#k^{1uAzHJu{Y-7{=~~%pHdq1#?g^q@X@!l4(OQCe#02IR?xPXyC7oTsY@i+&-ui=bN73^6)aZ0HerOx7%GScdA<+(Yiv);__C%3C7KYcF}x}F7&^N#wc6{DB0N~#8oX|4(OEaEnbmUkqAd!Cnwdv@jB1Hi&8$q93w3i`7qG%#F_7TL$VZrK=p9<55A3BNf(R4V!;nMP00C!Z3WWhE{i3&0$-t5Az+a`5lZ;Ipi|9di%egdEoIb{CUAvK7;dnvDHLJEiXjx3zr=gx(?0lLG=)*Y4W6aQ(pJIJvrNApww&Y|=NwF7I5H9ybI+=j($KG@pc%<Q3-AfGYW+VXrIB!^nW<bXfr2x{F$oc5TTFNrXSW7`sy0@{8G9S<sV>U<K_>q@u1Z`@!tf#d$fuAhoEIFp7q2yRP=rbDHb|jX~l-kQ~^cf;(ldsBY7P*Dt+?7X&Jb*fB+f$j`PLf9ws1}xL{$aIRt)49#rY8?$KE0si%ll%dlQoi*8W;zslv$GNK(}Tb)<KH7lraD~o!)9n10|#~j^X`4ofrUAT8AX7cId}p!pTV$awU{ks$yLVuaG)-$&#WI2cC_p5f4Ox%T~hWoO=#W4C5-FNtyl55W5W_oq?@u0W?O~jVMZSWxy&;vXv08i8wHn9fyqS*r*Qmwk%52v8IB`!4jvF<&<XN3hGP!O3|1pd%Xs~mh)+eB-tiLO)lSLo%M2Y-Fk*vyUAjv7L+R`Q0+J0mOSjO6{vBUE}|yxBgMQ<Z;%}`S5s;Zr_nDdxXYCUsCIKG2UN&~lExuhMniFcdaO+wmQbsqWm`1>R!RVX&gGNZt|YG{mF(zi-BEk<)pVWqwHF3(nYaby>od6Q5W&mW4Va`KO>y>-QKdlt?F4S&1R{^X`^D$vAE%ekfuM4rXK4K-5+2HJsB(Gv1R>$+HhFz`n~Sr{=RQaKzeyMXLVO*siv*5#B!bcm-C3n_5}O>1Z^K#sy6$q*DdUEQi`7}>8vlB!PpxF^#vZn967XfB;5Y7Me;a<bt&-YdQINGVN-avRmla#6>>J=r@86Nr<w>Btxtw#T4~aCl=?M}(QAqiM!&>b3z5~@z%x582Y!CM6f23AX_1ip&HV^iwPAf<}^03(7<N^v>Sp*O6n+HUuyY{wamq9~oFE8e38m7Vge1y1iuyc2IMzvcU>#Z`{1+HM(;LdkvXS-Af+nR*Yc!FWh-9<sAiSr>;OPFm@)tW?VpdWTmqo9}!c5dJPlLdnJB2Xg-JG4L=M}c@L2!h?27m)E9d><>2^lU-=$H^cLN5w9vg1WjFt2UV31>g{|Msa$&b((pzftSorCt(&Yz@c-b$BJ+(ho>G6@XjL^isa{hnL&LGW@$wFe6hDJ_im75WbI9MmXS{U?_d9;I4Al$VeWe~&JD5mVFM24G=a<xw4xd`XSEy}d*O&58VSd9(=%9XpBpa3VizT!e`mdQ@sS*;J%xp{w>@ieb$0MjJHniu{PNl5C#>?G&qbv{c7^tUc?=YfIF7C^KAa*U$c3Wc$@~LUB42Hwk`ro5KQCmalyeu#wrqn)JorO+yTfu^X=Fn8a{%K_4k<s2;j=PdhVqwxfBDNfqmpYi53)RubT;?PX9_zsri9ThWoZ8m)#|H@pO%asjr>q1PiC}(Pkl5&^Al=6i-1%l`H2edm(M(jA869WW^(7iOe(KV8-x3I4)!2x2bOqQn7lR=g*@xm=m{J1Eg+JGT~sp0m6tsWry^#uY~%OEKhba6I`Z2eF*1$Gd`6)YGW?qyJbC~fFyY6i8DLP(X>$2Is#@Q=Up?L@Ia5nijYwU|=WRe#3mvsX59Pnqp$C_GbqP*Kxg8I*zdBs&FDXV_;;)3|ypO|tozn%n_N-k&{83)wp|YGucC-8^^}S36SPCPR<g3FXt*dnmZxM9~xTVm60w{um#<{0;WwjhAZEo76H893vvQ=etNhc)-1k`RA2izDbXu(73m3$3g0irlR*i$<0;3*EkGqCme@>k-|-hT59lHV7f$pi2iG@cqJ1-9RiS83tMBa=e7w_QQsX8WTv1LUwxDa|~xEbAq)L|sDbY5j--$m>@Z|193Cc0sV-L%>_|aU4hNvL%<FCrW{kfaB=?47&h76K7*fyr=NyT=9E@Ms^g}n@myei?{Cgxw4HG?Y|mD?`<yITvH3;HFdHC7d^0~z7wV9LJ$63gk-MLtaSb+p?}P_rK9}7H-@MVhJwf^i!$g$mF+Bip94oY+vKGVL6t;|Nnfe6jUvz0i~k~hMw>6mvt^*l3Mil|z3fl^@LR8imLN8)Ql)CgDmx`%wZ^Jm!nh!(=Z8<9J{IzXQqEc_BQC!^dZ6H`B%T83Pf8wx-*bRW#DuG6%({A3n4CmenovZf`TclaOi-8=K<K9upT8-^1SW;E<#78WD$z-r6#yzu$RtS7q1z?ffKk~LWdQmzOrYyId@+px)pz-FWgmje{|0SY1lbPf1d7Co=;DdHOBl5+2|@78Zpg%QRJx8Y|7A+NL%=~?#H3x8cnRKZ6nG>N(_vb)mfg8cfPbe&48P?lr#Qogro29diRKjXdh`PZs3n;|>gD=mTs|e5{;Q(t@DhUI%m2QB!GH0e$!@Dr8e9MOpM#G$`4xfbSR+r<w6L@?-SwbVsg~+f`6CEPdT|3N3Q<dniDwh-rVtY5ITyXwW`tjIgWN7)+RnxX`zc(MRfq2zb6CU^%#a{>^HM`4<I7)S>tOAKxB|3xHzY!=eN`z-PHQs*qTY<c-h(PjiJ7(L*%*2$RMW*jOo>CEDEAMze&KC%!90d*0huN-RqC!PC$a28vwk6cpvKJ~-d;liI!wEo3d95HXH*A9b(&6MO{QkJP2AH}En2)*L(Ycb@eXe6@($WQ?*@zyn<G_ob#R52RL1leN~b4Z^8Ke7UxuT|)lHB2RRv_a#0@*9A<!L5&MxXYE?Y1%$4)VwDQj$~C(2z~GuBO%kD2EVY#bH7DXFKf!=gh!I{JY#nTS!2y)kEU&KC=#Y|70t%RifYa7j+haF&DY%){x1;>|pIoz17^EEsW5jH>3E!oGq0VU5Ajp?ai3O7|MD-c?VHvfCB^B~u>Dp>L$ct8@{BZvuQ}RH}MRV;_7IR$U`@ejLqSrP_{hpTZ(~QOcCA(a@F4YV4!O?t_P~zIN~F!v}m{b_mSOq&p|xxlP`$B(KeS^6=h+*B?4lN-9Xtr+H=3YF%yZ0BATq#H7DWl%uG7?5_yX^o9UXI&NPPhM{AKT=l(JUD;PxW3K^YxHJtgs{N*53@4@mMrXEHWZ&%TcUgNht!>JV4&)nF)a>?680G2I*Ep&o$t2A~Dg!B!91iX-8?d&@CWkDR!F~C<B3Y~2RxNh)O(7z+b<o{8Wd?_3fY1tkFpjtI-<|KOU(M@4{D}k%Az5YQ$PkP*>ASA5wdUGL>&^+DN1OG=7JPs<*WEO&Jy;cc>4I$?$KJZdb=t+cg7Au3cJn^#Dn>2858t^2(j)_?XF-b_QJ@3E_wta9TYIxQSXYhe?Wmj9Rs&kCNBxzz$`y*P76m|12@=jQx0@GBiCAylp%rnv9A-a{J_yC$zA1dmNos13o&gI=Y*TyxijdnK8rsXb1VSEmwnny%Jdy4v5sf>*D*@%l(aeREllmuEx}D5%h%1~sF5F9N?X~Ip>cfgpIrt#zOBK4xq;FKfvZgUE$SvPA-{qt{Cvb$90zb+v()t3JcWN0BWEE45m)A4~D}_YEqeG4vCV0v>)Ic2%7g^IhI$MEaB`^ve4xFijR%2lB!j#ID9IZL4sM3|oT1~EBwke+8X<y&!%eN`}r1xU41!Cyrt^Fkkt(|O=$U)k^ce6FW9=%qbgf7(Xai}(_j>M8X9yeHJ%?i36i+U3TFOPzdOS4kXr`f!_oU)SDbt1sN)^{<oCoShi_0--AeHFj~Kxz~puXT@>B^@x|EOAM+uSs2hMc=cX;b|^u^!`!H^4D(^zi}v~f>>8)DV*7y9X-kcwHswdsNHHLhniih#S|a4L}^9_5ZC&LVH={%EV#&(Bwfs)z6#8eWm2PSiF$0<dfM_b)z4P(8XenEG1Bx|oIfK4!_}@1g;!nk<$P54mhT}7f=7du`2Dd~Tj8v7W`uQh_ja9dv=4xk7Szs@Q8ca^^R&!xg&$1ea8fsG9|u?EK?BviMkd~R=iY0Np8kdV<l*BtpFEX;rIy2HU}xQX=fR_=&4K#suJ)l_vtTuaf*vRizH36MLPOR&6ljPas}se-Z(01RY~^YZG!D5uP%XZxRzwvAU(Bwy*gTw#S!>8i&K3gqd6#C@y@9TNt1lnmn~9mcS#bSQdJ;4Qe?rg(hR7cup`#vuJt%$Qq9tKvi7%oMXN;mVeWOQrz5DP7?!zZf-hA>li{4s_sH+wyL}jGuqH?N?mUWEe*w{VsQa315pIiT*iO@^pNBBO3N)RZ{MPs_nBjiPu#uYE>&q>mTl+X(4+q43I1VE;bm3hydRZ*)9=ejua<uQD{gMu_ma79WTu^HSpnkpHrKBrhtjs{)TIyVaem(rYl^{=a_SFop!t<EKaNCMX%Otj)EWcfJ=p$tXK;_!#0mAz>@1NB5onACFCRv;7jQDKn5r_J#7`dHq-qBhs8RdAPmw9*(>s|~)c9JfCAx3b~#Py9P>ru@D-55r^q@!c|hZJF${?HlonT{kY_H0)|$uSsc_o^AeZglj)}Qy0W=;$kDgTYr64Cb)Bijc9M%OyKJe%B4b6-*1ru(<J3TsgZGdU;cRw=Uz7Ei<XbMrx&2`jX!a~pXIxB3)6LNV<ts7KweJiSzB9>8cTlh1S1+4sH_;TcDZ~G29gk~sr-v3>#ug}+tJNW(rAdIK0>2!j7&npo?&6blRmdJ3WLV*)^+6SN}6%eyKybK8lanw7W~B=WZC6<iR<zw)2>VOcU^0sK4c?iZ2TX}iL1x',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<O_SS3mf)Sg0;`S<fHKMYklo{^L`|qm>b0WPl0uU0-YC}~P$XD{izH|ORH>|~^PcB8w~g4C+Z^Vwr|H8UdSm~G|H;13L?)16u_X7zC>$;V$b6ak^5x5y?<?=!yZ3gLESJG%d%n!3!%cFL2FY}qmSr%_SDU2B%6vTz&Wrr2Op9QVCbP6CgKQnVNq^d=WmOJ>d{brlI#~v9(`CBOwyV*-d-r<tB3}WF^>kYl>AD)tx7D^tq3UXr7gg|XQ2?yhdA@x1A)Rikych&Y8Dpc%v`S~DewBWxmf5-1TW^<3dE1j$)9QMYmilz@bGFgnKSJmA_bRCt`a3Tf{081xuwPq6id27<i*1!H&C~XLQ{;#s^XA(8tg@A<R?z^n^#wPzEtb&I2rycv+8(_7G0)Zm{4q=CMRK8|UM|xqBGsVlB;UfA0Y^|S*)FSDHm!QS@1DJR`~20*U=oCzBFl@cx*kIB&eP%h--f+!o;`UA1>oz)UI72XCor*_DjWwPqVHf)t(Jpivw?|Eu(5*=_~wfb&9|%N!=EOHqu&kgAKa&3Uxx!5;P6GVzSsigAmpDH!;_bTpBGXkUAr~D-CPtvLc{0l5@x+fhe}3R_XtYe^m;uQW3W!EtGu|3)1t_W_%d71qG^#_EfELlhiQLI@MdY1WJ?%fSrrk!9xd~$w21ojHJb-nnXStzSx-~^JO~8FsE}qy@HQp-Ek|$PzIcYM32gjVq(HRmAS}xz_ayujz$f{74fFX9s3YH29`I8x5LaB{lNxyg)Yjbs(qapIvc8UJEejgwc@fZa6t0t13ad3=ZV@^UeGr5fP;fRYvh_R<=1?V_;amO{RCz#E-lt*T(}mf(Y~T=jwaQ_wXQdzjz**ga-h|U-UIJ+O4qXYZvT6Z?z@|1Z?$oN$iCCx?Go-q03nYL_t9{{c;ZK>N?%8oB#?z*tjaSBW0Sr4`U!*Z|>quDgAUMxw*8*GFG_3qfT@mBpJx%cC0DizwrOYU+(p4GtZ>S=bg=z@vbbNGp{AF-Iz&~fs!kXs!Wd`j7v)WWtJYPmdIx5qom@cAveCOe#sLHBk`j3mF{txG;|M2+i{^K9Y`wXBAR@<@)&eI@WC+9#y@a56L4@dClG0+vY*mqhi(q*2^pv6BV%QDqThyWcC4L^7ke0}u0W1#F=a30R`>2`&C57(v{CZMK+_z%lQ=ixGEemCMa>cU8e&gacCnOeYXS||3s1dgeE8#$`^S{2u0r|GIlQy2^u8evu7Vz6QGP@IDAfzALy$moM`gquc$bw?j?2aCdk1FUjzfNVYLpRV&NLXs?DR~?VbX_0NJK8>^SrMh0G{WDmtBW!sSxs5h)j#!qCAfk_s4^45V@*w&yS#DEeq5~dPO5J>oyI#>`uD&+j72DKa1xnr{;5<<{oex)8m0ctiGoH}!`(Og7F)Jg)Idh2AX_O}P5z*!2ACUr+V3i}GbWjMD-=qaBpB2ncKwB5=wZZ$e$mX(7?y4(oB~=BDFvt06n-yt;${b|$Ww{q17^LMQE<xmgs?j{HrVHE!w}8?ZIKWyt3P@c~ft4l&u;N83{Mlxm8(5~n)dF4z8)Rd}dmPO-@7}!*1Ul@afLoH^5F(U%pXFO*B+~3iyJ0U&U_Ld+(K@o|@bBh2>VxD0B1>exN(BZ39JPY3FvA)LxXV<qp=;qmaM@?x#hVkM{Tc~iniduQO}5nn|5gP|XPmCzhr-Zc_CHVuLGVge*iq;ew&V6jinV&lwc4FWAbkf#BidgCEwglK5QdCnsc4hYGOoLjDo=`{(RMR~Nr@U5^Og~jZxGzS|1ta)W_b7n2?T_znp$pL2s%vyK_1;Qf*viS*mBDds!S0^oZDB#ni)-vLYmq18ulgR9yrBH54t(kJfjrS5SRn2qAr-LB&%Y83F`<LZY?#Mg_FJqETANv%$C_Yg>wh2OW+aGObLy`+~`gY#2C+)vxsV)h9IhkX9HSXaS1DFT`jLC$hG>kf~C?TFDs`8k93`10c70mN2?roalX!`GU?cB__j(uM8`0D+Na=)TBHC5p07sRb%uN>IvfO}uli<Kqc!mUIH`zH;t&SGVLupRlSUY(Hogc(N5SKX89tY(F)Y0%<i&tbkUfA;o#EDV0{ml^U#$<$lj&tXpA*|pXC?5cRg$fdnTXsIoNs3rDkfR%yxz8*W@yTQtLE!jNlg`S+?hrAdJsI|{u9>9Wtwj2ec#?CXxC77NgaU6pXSBPGy7$NCrmiCZY!II4gwLv2GVy}Lj#ACn~{_ol-1#wl~?72Ml=Yxxp<ZplaPOxVa=A`E)qauK&NZz2vI%EXM$2^mRziXVzOyTyEJld0I>na3q+EFbaiQ1@_BlZt-FZG%;`kReIbJpWMRLQ_bDEL;}@`<0_jYY(zewU8Mex>lhZSNfU&{2>B)l^*mj{|9K4+8O#OJ3&+r4`3wSdOd`dgtyn)Lg1ysU0f!c{#BSA85OlR8A*R0IK#}FIAGvp>+<Dt77JYXz_X@}EL9-|-;v${jt5auXK3p`j7nq(D!4*R|XYj)&Bq>~t#BCLN9_rGokP&77Jr_1CTdh`;`YR=~t7RD_kwxm)IXeIDdnMPWqI(#m-$zi&H-7@C-Er=o_Dvh1sUy=wr8gI9!Vu7=>!luvd!!|riL9_+J`*_p$!%VW$MGqoZIjuqQQO#84TL}y4sNe1=V7IvrZPq&O?u7>rQK(1SSOK6&^Ty$*fW}1vG7fN)^AviY2E;)_55C_s0e>XY9V<zru*8Fcck+wgVG(=%-}E}CM3?y42=qj--gu{LcR^T77q@aI0`^l-2l+^NCyk=Kx~=6;J_XyfDLg306S_gc(V-k`<&T%q-{l`Il8L=wJ{O|wMia9uDNRiDN9PiYT1p6^p=YX#7zD50@@Wiou2{{F+<H==h_?hyymC0wQcP46cRK#+@bGL97zGGuV3M+&pt6=#b#aVZra0aMSG5*!a+_k%Lt7arr<WZBBytGM*#<X-wh+=<;6tTrBbqc=f}$;>?VAX?i^l<7Cw#8Ql9n<_VC<=|+%+=O2#-mhyj5to&_1CgqP+1@n<L`q&lR=(NV)yI+gVx0EOl8|S9g#wU^EQAe`s0A)*S_y|2fpoB&jrH-h?DD2_h%vSHBDCyo^Z2(U7}$B1+kDOH%KU;qSN|IV&w3kd9sGQdW1L6<+eP1-@1aw5S+hZKq@q8l5C%ispbdoLaK?=~EO=owIHy!&y|-CgvSrq8owPLj}YH6$(2wpS!sa%#Kl^vi5<jPwQ%kj0`tu)ePcq)TyMTDJR?ylX=CDP0bZN7YS_J9!UjI6{A`Y(Ds_Nd+_4G?798mAei6#sA6db(|QwsRQ%4e3n;}3#GBE@&-Yr;wPaG|D-f^GudB3-s2x;I18{AAK$f%-8sq?{CBYm7D`bb0!~QKZqd+G5V+6LnO<ran=wJaa%SCeh)o&-hj(5OrvNE1=nv*XNza9i%9{-Nc;6RwcBlG!D@Y}<~Ue}s)VeaO5<F3{Fx8Q<%Frv>|`R8B|y6kmxH{a3{!jF;gCyrkBM3%%mSrQ@YC2-e`LH3-e*tKJ!eHOj!1l(i?20`w7)zDbeYSGt&LpT#WI6TJxJ;49{9sciY)~P#~c%jU|Gq<Fn0ffLbiRHJJNCn7g4dAT5LjaevcrAf{43QH%Vt$pwqqYgp)|A2r{^4Oi_0d=B_Vyx9qkKnpzsp3`#4}1h$Bi}n0G?5urf}NG{7vvjPz8~ygcR7cNwM^glahgiqDB)^zO%lC6)jmgy2B}0kWNr&jVTQRdWvLP31`%D_9{U+Ko7)B9}I<-NSF9-q;!9OHXuazc8ex8`oW$_X4Q##WeX-9ED$H8F5lZ*GM_VveH7A)Eptp}nJrll(<mJJ_Q5W}glei(G7o9~4IntGN8!WTMpL(owCL*!%!S)J*bP5<ns@M20E^lmh5OSsSwaW#b$=kFKMAEZ6#aU>eS5x4E@EW>5jWtLbXhx&u%R*9p??tF+cE=TMLN)_w`m<6-O<W`pu|WwkmdpL=n!Sdp1moX(qzzUD@4Y<y(DtgQN*k@iH+7IwCSJJ_C4Ry+iz^)7Fgtl7MLMN9^*Xx*gogpjDwHx;wE%P?m~uLGb<Nfb!_8^I_W%~lIj-^r=rUgFjue~(imN$TzSuEu04|F2#eoSs&{nm{7SU$Wr`i!k|uS&UB)V|*1{0!Fc>yaLYZ$t35+WC+cf8R)CT)3egVn)Iw-d60wT6o&@LT+_<$0^Lu2q<q|?g{?5?l{z(idKq7pVLbnXiEKxJ&i#Ymi25ZCmQi&*c~j-quZ2nzQp<d{+%kQlBKu+hNq3436p&dPWWU9;_2ya;P^c?5bs8KJ;Y*N9{dl$e!h4}<+{4iX*i&IS!mTO(PstE9LjKMw5QK@ckTVz*~Jo*CMtt7}pfa@#RLClR4g9N^K0=MLVVo|a4=fJIS^pVhuYQqE%}2<xc^DD}gQ;+4BSqT#w$zvMls5tWqDKF*ZPRU95dUQJkmc3iL=|CdSQ0H{OO6JX!RAa~6TY#^|E%pRgn0*G^+UeSaw_somyk;S}axloUEpi3BF6!GNLcmNS$nV(1DeW9EE$VFTU%3b{^QnYx$bX{g3E+Qt;q;q;^%&$CY*7Vg)%TctZZe66E?4s8Y?ns_*Pby6;cSxmJ8(7m9r6F<}V^*6cusw$&WSe9O3}E+_>+l<-;69Ow(}96&`?)ibJb!xd?VA_DDl5VDbh@x<qS+^r%Bc^Q1c0o?F4mUoJYtiuBax_`X6zFH9hDFJENlOIvw|lz<*YSFGzS+s-2t2iSGCBchJlO|oM94F3xT;#nbF=80g#w{K@$s$t%P;2lLO1do)BWCr0>XK$Wh$|)a0Cpq$US;m?kT?aj`b=rx7|?MorrSnU3r+*3N2=TZRbRhrmAKB4OA_<kN6?b|+LUl5%lSW<RH0G}$<|*`(7tFO!wcaXS2b>iJFZn%$B>RtNj2s$$f#2M(tB!ytzvUvY&4H8R061?E}?GhoU~azc=Kw~l$_I0=qO#!MKo&&arW8ntxrRANBP+H;zp)A=-8&X8+TQU7_(ZIx=cXh@gdX3<FRH#t%PnHF789jYX73BN!+jI$c4{%woVIU6ShnjQfLZ&`N+AS+vY)3L6Vb#ZgeLayhm|FGyKeM3G?R?y%9_P+){(2n224P2F_w0alr(7+jT3Wd|fEuscB2xnl`RD+e&uuvqbCsDb{XX1A^f4Nl&&C}jgJ1g{Es^|7n9MS(<;)Z{nQ`JEb;fj;xzEd;1V?Dbco2PTnNvtWL-zr?WXEDotj*Sy~@Df10K7ni{fT#Hye1lUkye!av!^1{o2iWABi2Ur(l-ck7Z4DKlqI|J;Dx-I15yzF`dY+Mm?1dFmXigE&VGl(Awy-ta{gpZQ@2hteOWrXmXD9Y~s@J8imf#}4L)I=Wd9T|_Xi*l|pWkZ|+<}kQ+LY@oSUJDk(O%TK2=1(is82TCvWA`<$>mMYeP&uRRHMk*h)pK=Mg$l%)3`IxsZZfk=zv>uv?P4&V<icYpm;67R)E!DP1c%@C!{<%5Q0Y?l$cfFWCzX!Gg1LeE4H|<^Yw6^RIsrN;RWc1497oI50*yLp{g`+Ww>%q8npttHgu2&0`idh077=mHOPCeC0aE=3e+-u^O|FT7{3EM0WjJNJ#}z&N_`BmyXKruv~PA<*t&5=HcmKp2|CQQ`l+gBtK?1S0j|y(q7~Mz+x^mRnNItTWHfNb3H)bk!2!Fbrc7K+Gscxp&+dTaP&M3E3q4ENs$0US5UCTINRM_963W`wF%`BcnEbJNJ#yX&Nc;P`$S-h-8`6CZQ7e_I-{8!PQHv|kB?F&aUV|!ryMhIRa=}Jwl5nDwO9(*RoTJE)*3^q@CXPWNTE#Iqd@tt%@iK*l1_fu^B`(rsegQi!u(FAJkVW4-RKc3>eVt?MEFi;tz*v^s4pMbtqn8Vi^P+T$0MiTL2$r;i-5r_U;)B&?dafZ(b`Hx0Y2ss1uEZCDb$NnGf}I?vgWeswm6?cBTy+vp8Z~;XM7e)Sp1sau;2d#@{(l+_I|BRWB^0VBn5pBMCfJ%j8bm}&&DnOfDZ#?YSVG05joi?nM8B99yKGmX!L?SikS`I9`h}l%LlcBS751*S@x9YR9X4_9fM|Z)4;1qh9~}D9zrmGM!rl@m<uuFKi5+eB%Wb)^%p&zPx^(5Dxt>gMy63f8?{0f;Ma*<?XJiX`d$V-)HHN?%`UiKNd@%`*+TRtQr~KorqgWIDr!DqvtJA<9bP2H(V7tEs9PiR<OagGX><az1PdDPEeMqrcLaPgRTUKNoMvr<k6X8Bits+gMf?vlZZ_tR4&Fl}Piroh$doY`NB-{^l(HUoky+Gk^Zo_h<jse9G7NuPPUqD&{;BI?%wm}&ds3y(a37oF!4pKpN{(b;0L1Nr3UND9253mKx26GpFqe7{>ivMeC(}dlPWX;B|IB78Rjx}Po9cwFM+iP<p+Qn+x{e0Z!mwWVp<GN9{y47+D7E^S}L|3M-2I6;qG%_w_<>Y}A)zsVJS{@xALd7q?8uc~iX>#;6zJegM7;H%s6#WMNfTvzpyhs^yI*`)TZku;uOlvuKN-0DwS{;cN{4yOL5-n@3ox_0Sy!TD{?mfzNq#YrnUnFR&QMKE^_B|K3odWB4^`_`oj_M-|YV8EYq-Z<F&R{of9q)i*=8eh|laqkL6DhEwp>Y3^#_IvxZ$zl=?!ROk-~mYRh<2abp)WcTYXDKh7u^n$gVTfQbp75NJ}#r4F0k`;S<%+<QS%3Kj{O0RojV6p$4goAH|Y4X86Ab`=psIJ)<D{rrS(Ds;E1V5!QrlEpHd+Y*09ladXY3RCkw|pOj3FQqEPwTq*JZ*$O$oOcSg%~A3zWqwy8bRP{UfC_6L3wNkj=U4~g$G{6l}B7Ata>G{Og~PrvOq0BaQ;?qL&<lr*ecguDZ)T&|;w&=n#^U2znXTSOdt_N`D#@2akEYA_%v7&>zT>pTspA!{6%$h^>wSVYn1ZxOQ?x;TYqoH#m6fwZ1y7hO#;zwC0Q@HD5nMaJxPpqvSH!W^SJHUY018CKSzVL%2UQ3#$U{DDp3<{C1xki*DzITA{MYLF(Q!boNLppVAeLv`Ppxri|)!&lcCbsKQZ8MUc@4-m+Ll~#8xuh#-2AQ*wn!oHp!-KzZ^eH)mLNiG3)R3i6`JGf@-tv<DNBRiZj_I9msT^ZDCLZ4XUen+<o|J<Aym#$r26>V2X_55<w!@Ib(g<CTh^6PXg<<B&+1go+pu2L+rBcM?fkm`L)s^K17fCJx#Qc=_?y&|$cdFEcASm$YfmI4R2ooNcRl$^}!@Br@9DFQ$9VGZR}F_-)Zu1Zwgn!(=$uW-kk(xxPx2TMMX#088C)yPtNOb*&2l?eWU*TMV(?zR-%y2!+h8=bf*8H8i;s!>CoXa+@JQLNjql?veYQ&2-lb{`!n*z)&kE+iurx7URE<my3eN*wtLh!PhGkteHX&NOwH=yn~W0L~tvD8Y&}&O45kaSu!&15f5Yl5F%;B7ez9jtOV1NOs8p2<#0%j3AXBxEV(^Jq>s+IoAvQH1k^;i|$!>Iwh>ejbJ@Hn_whu`}{cQ3py~s8ni@MZ`7#4wVV|lThD3N{?ua4u9dCjJK18edx0_B{7ep%YV%c#rqQIa4b7<U-vIbD;A6Gujx>!2N&4$yLV5n+y=9$(Y;R7!M_0+$c5d^SoJJH#TOu_^Di%2wBswExYZCOyq>B>Xx=iMvNPskSFcBWJlnEfu=C^v!waqP7g}Yc>q{FWN-kqS|&1lx;)%Ty=ZpMZwtMi{|n^N16i;1RwO<UE{I6F$p?a49KDTGAp*lOLrkE|LR+itL2z++nDbeE7vCzciKu23m_%WAeK!Z_t)7EuKn3v-E#DoNBapznd>jx>g15j@q#U`X>*r+9=k3QNknHO~snvz1I2v9NED&to-MQ%0`kGTeQnXQT$dfAi}5x6j_h@1DJV7ys$i(`WX18oPplAlQo*W<^LoN2@)aLJQ`Ha2_I#T2Mw09^;K*5_$aI;<b^3sRd4~<aK6L$5qeX-STm4P>;<VyV;@FSNLws(c2lCMw8@5;L~p~yUx-NA?4LOv^#I61SgnSijQ*1GtCseg<9xrM`c<E5u7id9BTL}77%hLz?$+=ctErQG=NZ7TJfmxv%q+q%-RB<<{b3~hqEo^=Y={48I?03q$P_@7;(s1o203*loWcrr|Di;S99nMLk@82v}5CLaYl+jeX`e}R8dA)|1^|OK!n!so!J=wm|n{-4AS+dVr)<c=WxTMWZVMN2fuA(*+Igv3$0R63CzTy{UzrS7%4`;b1;Pi0{gd1+Bn_}(4C9xJXPe5Zwmupn0QXJAerQ_t!~<-Xnvp1n97wXh#E~o=13vcmM2=P{nNuUtW_5@^o5^Vbj+lnq^vI-Vn)N9DS0|PIy=?1e%4oQ5EGP91T_RbkmEq&P2KHdf0@CvqiZ;*CqB2m(KW&{e=j-1-SAA`7FxUPwfG>4ZQ$^f4hilU3xLry8GADfD;}DOE9|zkOfwL-vrC*>G&u+4j?b6L`Z82Osv%q&E8)0{*{_rLm;nMalUyYQ=F4k4TsB5h*HArPtzhwhJL|*=OHhVq!NA`YfBo0>BA9KjfBDP*sDjJIU;j7gJC?uv`@dWV|M#E%-@pCxm;W^l*1!DapED=}U(WyfKQoA_{^j5QPc?Eg6&oKhL}!C0Nf(L-M(~X+VAU(v;arcP{GgX`cEO`$8ne=v(x4#dk2FnV!)r(M3<q?O1g*CehVl3A^C*M7r^P!c7>t3%YlN`{ATUnQ65FkLJ_hDJ3Y?A9$5xm-r#5!U)y*d>I@LzfSC0!b$(ehln5CX$Y0iQgT<2SI5L|2_xX+YuI15>%@FgZEJNoLd9p&AaG*SN6DifZihQ{PRs<i5~;)B3W<Je4cL2iqs9vFd5LCmU^%g9^{z_3gpuBYx*dM-p$ZZ}wIw9T~o_v!gl`u06^rME{8GJj4)O=hTVhP%=v$~e_<;z{Axu)d5)WFyxPzjLw<FRgg_*jSG2hgKD0Gd8dCl>m4bkrEqP_^eVbA;bdQ(OZ!=nltCkEVhevSy}u!yO6IRO+Il2aw@nTKt5|YH|`pTLm83MvrVymNUaJNJw$EJb?aFDgJibI$xR_nJkuO4P6n!0__riYk?~4e0hU{xy-Q2bgQd1oF{Z^@r?X8wUCqGKA!6Deu;@Do0AMNUl;t6MaV8J=Bbzzf3XjbeAkHXntw;FYNpkf*R`0cEmTcZ@#MlWiniUBTgdhtzVt`au&4*uyw*)4bwfw1p3BRHbygTW?QnDnB;xHpNhkg^_(Gii6d=$QQGuo^#>e0%`ux1}~JL9&xx&+;0#1EyoYLL<~&MzBYJRtcf7XuG?s~ZunaSL|ezjm5|Da0M%$wJLu5hL|YCl3R1?E)tkCKoULeL}QHJF;PSiVibESv-<50@VzDG2Y3K(N37E@L1TT98macM_lK}8*kj^tvO_P!d)dBI`#a>%G6y}5<5%GJ=Q9A9!IRiG51;~60cvqD613`uEOnX+?hwIQBY53zBWE9O^A`f|MTPsu~JxeXZ7Ha7_U<DQ*`Tf@^8J|p<A!D^8nO&(Ca$wv>cOe4U2cO$~9|$wRdbi<epRaK1nm}E^f1@lOJ5)im1TUZD5n6Oq_cAmV~_p7`X*!4A#jWfGw?nXT3<tve7t^6x=>0#3CJ%W*l;CVmdEapgk)lC_s-7b+NrN6JWJn>(^@5;|3k|{Sf&iJ|b^`s0jZd`ulOm?K1yJnPKpB1!ty0u?pAH?Bm#Mb&gMq)T#QVt5;}RG;p&SJ@*elGd0w(y7M&1>XDqr7+GOWtCExx!6jHTd&{L>*P}#5{*5&dc|D+5Qa#p#cV6u#tHGCWX;Z<p<e{LCmZ2qhKn_b7p9T-d^dNi2I5@hYF}J6__Q^i<_=nLD>^bq-k$iq-AVd@uTaxysyEd319;t7uCOFIHHulLKkZIP;VqP?ms-y|-#O?>-!5-b$cTZl$PrrS2!dXbou~i_|JX;&x4Z+FcPdkD=x#7t--#&>?zIpQBL)Z4vX{$gLn#s)1X2L&y^Xlb)i%p|_Z>!8YL6}CNGiFsFdM4mr;CM*atn)d#t-_6lVulP(s1gW|bOi(4!B;>m8ZZakz2WY<^91h~djU%259YFk0I%;T?R-%!&tvEj>KoAavBS@eVhMoX&_3Nvu8yG)+|Lw4a;KMQ3c%D#XE(N~)Q0C+hnCo1cZ<dT)KVp0*Rkf@38sI0#G#uoaYnJasWJ3^+nkNB`Qk8xSG{eJRvWxds>|Wg=x|i#qeB>BB=6wW@UZ&rmy|)beq#qkXzlYN123H6aT|zyI&Q(TKQuwE%klvLQZV=k9Bv@rHpQ3DM|j4s4up&I753yvOTqJz0rBDTT(nzuXU*Hg7PLR(aS=UvQwQh=^SXxp=UIZ5DYqQIJ8Hw7XCJ`i0A(MTvh=DR?w|2anF5O(n<Dh;xCISt%^*rTMfugSg?N6^MEDflckGJz!m37(%}N{ayM$DzS&_G>$Ne;{F?v&PQtE_Y=+gEoP}X1!b84L9(v(WURBYbZ=1CLE7m$ulp1yby35Z6D7d|&g()~67>5v{kS}<h2NiiN}JqXWOdp1wd?|fkdt7`;OYe<<vo!iR<Z;6<K1(y<<(2b}gxJe8|t`NB>`VnIi%hm^~5_)sK!e_OlP^m6Q&^OGB#BJbDE?YNaei%qU4otJb7ETW!Bct#K&fMVDfdt35=gA=ZO;@@UR)|X%>YGP4!*rX?BC=2(dcu0Pv%G~Jd>MML2_Oa!Uw7RS;*MRmHsA2;G;l>unRX~oEF6wzOLS7jc6`ZL>H%X3z|L^pNEQuuslh^9uQ(+zX~_W<@^%nBefIn3PhQ4<c=PJzyJs(-PT+c-HAW*8>;wu3e{m8&dGTTr!WQ3iHM)4h1lL2^2qQ@ULZ`LT_+R>bgVlwjzK0k67b6+S7l8$(tY#SIWR-mV{Pi<>52q0O{o8j>Uw!*-|2sQSd{q4x>L6i8CZVd8=!8Rx5xyvLf3SeniG}Y$^AeEu{IMgaY~CUs%~E6{&@a5P4fL)c4^tAOFbUh;B{xkd<R?P-z(PV~uAOWnx{nJBLU_<?N<20g^CFGQXYU%2^32~*nY;gDa8>}aY@(vXWX^!VyP&gAU|xq~*Z8bKk~TS>63Yu&=You5_|s3Qt%DG`2GbDnt<5$XTZ)&lYB_=8Y(1w^`OJ?M!`#}-;glF424T>xOyNM`pLH&ayIxq+xr%nKj=Q$fT1?%G<iTM_vMCT~$lOLg5PC;#5uWJ#G2AyB{qXjSK0r4(b?K!mi&R~_CSeXurFa$7vhy?|kLKuzJzd~12%4X;JK>blmr4in+<k9FJGZ|-I%3!AdYRMytW;T(8qz8*E3cemErl{i{UQao>)Xa=>sf5_NUM$f42Q#YK9sjZJQG4*ku^n--dCt+gIY7vJPmoySZCxZ#1F00lS)c_ZO7RT*~;)QZ$)1=ksaI%_ixDV1Bccf5=%!@j#a6B1|<7sxUce>t?91v+FbL!kJh^XE1ogUWo@4Y$~~%Tq;-lH|EXfzjkLaRhk{6Wvh1wqU?_r^WDQxa?eG%}qV5h;-ba7Yc63mm2IKFr(-(qakLXh0X{vSK@+0?5JS`?Q#1WI>EB+#+E#*ewL>lWRAHhn*fJqc4;Rdd&aCcEiR)r)yoTcxF<zhA5l7r&zI(!#AR5{;XfcM44aFcCrtHaj+A?6&M936jsIQZS+VeQXVPh(BvU68Qskf$7%UGuIf^p3N1F~qo`@dwA@?Vy2!mB9MX^AB~`9D-_t;P<iKEF_*0P9+yHwVMbO1lqnpi+PyB+ah!dqEE>**(B!~88E|(9ArNb^AO``bRfr0$50P^2;_IY7_*lBPs6v*;*%#Q-#m+7KY90!@!vqQ7lSb_(UH<>EJwjQRj3!@yXSv;22uobFA&l~xl@zLdZH9fX`P^@4N5Y`qAVh?5I4!bT@uFzW#ahq3a$<%c7PTpO|fCgG$(JKJ$d&me)%f?{`t$NufD%&k;24~0xLGv-;_JyS#6OFW+HzFIyn>h>+vA^bdR8x_lJ7pvQebrAbUHjwL~vvw;vr8o04@*xUwjvJ3N@!Mw9QPWMi`2VD8<Hf?_AkLB9%>_-556I3|u={gD!q+c>lli}BOeqlL33hG3Gm;s>iy%-b~_+Ame<(Ds&)L0D|Q`z{#iJe>CJXf@s~8?KkIo|oChqH6AN_m$9$()>I2B~)EGX`1zd(NaTUap<TvM-O;oZA3LgZD&-PH!)ZqDkfhI{|0QfO}mp`A2)IWHohH$M`U(;On`=F+>Z|U^&|97dMtw@iRjS*y$XE}D|}rLa`*<gaBs2BRN*V1XBnF*k$^_KWhDD%j<7Y6X2sR1k3_=rU*Wo^*TSdwqvuJ2hMh7(qe%{QtG5H+7TP_EyL7a*1b1}RQ|DLj_g<vJID3ei42&HPe8MdsBpaI?kQ9rBdP{37bkz{I*SAwxiJgVWFB;oPh(1Y1b{#G!NBG8*!sytSAgG-Y2$QD5x&k?62|)|_67N<FR7y3xjIRQH7_hLw$y7^}$MIZFXfwP_g2i?k{xq$mNg~w9=3-I>IY}kk;cprp(Mm1UombLI4^g&ahD~&3^JwHIl<o`}-ksCHvMBG&X`nZLG_w`(2KTG%5shH|$t}sQTvd=mqs~)<Slcqytz1;y_&)xJFqb=sY#wfyy4MEp#UDCT2&NoYws(d%ix^rr-S5bFafchl7eLiHWmffQWPSI&-*V4;9SM4;0EteDx`P_jVVK?k3pen<g9dX)Wa}ne@zs!Pkm#=4I`Wr%WYC+0?}Vb5$kKabt?>czxC>f55syl{FH`kvE@`UEC0zH2i6$aRSal2{qKPrtb?shb&dQ0P#UPO5wkF8f<kL=ge-ro1JE+a>enY^W(u;WNwla)(%2<=cN!I+5h(Ccwo7wz-T5(Ak'}

EMBEDDED_SHA256 = {'audit_snapshot.py': '75a6118be7c78e17a4fbc1a2c2fd50ddbf4616bdfc5750498a7afda7c808b558',
 'corpus_runner_snapshot.py': '147eed707d4f425d88d0579befaa1c2e43a856a9a2f767ba4785f083249a4c98',
 'crawler_snapshot.py': 'c5cec4dc728be1e95cb316a69a3be9690802cb61587adc91357f0414a5311442',
 'diagnostics_snapshot.py': 'b544e20569f23328c4df6e6dbc07cacda9f9fb6921bfec77a3fdaec307a2bcaa',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': 'baae699808ad43533e87fe992302adec5646cf648e8a2b3fe4057aadb8a5afdc'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
PubMed xác minh PMID theo URL và lấy tiêu đề/abstract tiếng Anh từ DOM; bỏ menu, figures và references.
Trang yêu cầu bật cookie được coi là challenge dù HTTP 2xx. Khi cùng mức QC, ưu tiên nội dung đủ thay vì text quá ngắn.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.

Mặc định `BROWSER_TEST_MODE = 'failed_only'`: sau khi Requests hết lượt retry,
mọi lỗi HTTP/mạng/timeout/challenge, lỗi trích xuất, text rỗng và QC `review`/`rejected`
đều tự được đưa sang Selenium, kể cả PDF cần kiểm tra lại. Log `selenium_fallback_scheduled`
ghi số URL được chuyển và lý do. `records/` giữ `selenium_fallback_reason` cùng lỗi Requests gốc;
nội dung Selenium có QC tốt hơn được chọn và xuất vào corpus nếu đạt `accepted`.
Chrome/driver không khởi động được sẽ dừng sớm và lưu lỗi riêng. Selenium vẫn có thể thất bại.
`off` là lựa chọn chủ động tắt fallback; robots bị chặn khi `RESPECT_ROBOTS = True`
và URL không hợp lệ/địa chỉ không được phép vẫn không được chuyển sang browser.
Sau cập nhật snapshot, dùng `RUN_TAG` mới; có thể đặt `RECOVERY_RUN_PATH` để phục hồi run cũ.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
